# 🏢 ARIA Enterprise Intelligence Platform




---

## Architecture Overview

```
User Query
    │
    ▼
Intent Router (LLM-based — 7 routes)
    │
    ├──► Analytics / SQL Agent   ──► Enterprise SQLite DB     ──► Structured Insight
    ├──► Hybrid RAG Agent        ──► FAISS + BM25 + RRF       ──► Grounded Answer
    ├──► Forecasting Agent       ──► KPI + Funding Trends     ──► Forecast Insight
    ├──► Anomaly Agent           ──► Z-score + Funding Outliers ──► Alert
    ├──► Investor Agent          ──► cb_funds + cb_objects     ──► VC Intelligence
    ├──► Ecosystem Agent         ──► Sector / Geo rollups      ──► Market Analysis
    └──► General Node            ──► Conversation Agent       ──► LLaMA 70B
             │
       Critic / Validator Node   ──► Hallucination Check + Confidence Scoring
             │
       Telemetry Node            ──► Latency | Tokens | Route | Retries | DB
             │
          [END]
             │
       Gradio Dashboard (Chat + Analytics + Startup Intel + Acquisition + Telemetry)
```


## 1. Dependencies & Imports

In [4]:
# Install ALL external dependencies required for the entire script
!pip install -q \
    langgraph \
    langchain-groq \
    langchain-core \
    langchain-community \
    langchain-huggingface \
    huggingface-hub \
    pypdf \
    faiss-cpu \
    rank-bm25 \
    gradio \
    plotly \
    python-dotenv

In [5]:
# ── Core LLM / LangGraph ──────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.documents import Document
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from typing import Optional, Literal

# ── Retrieval ──────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ── Structured / Analytics ────────────────────────────────────────────────────
import sqlite3
import pandas as pd
import numpy as np
import json, time, uuid, re, os
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field
from collections import defaultdict
from pathlib import Path

# ── Sparse Retrieval (BM25) ───────────────────────────────────────────────────
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rank-bm25", "-q"])
    from rank_bm25 import BM25Okapi

# ── Dashboard ─────────────────────────────────────────────────────────────────
try:
    import gradio as gr
    import plotly.graph_objects as go
    import plotly.express as px
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "plotly", "-q"])
    import gradio as gr
    import plotly.graph_objects as go
    import plotly.express as px

# ── Env ───────────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

# ── Directory setup ───────────────────────────────────────────────────────────
for d in ["data/raw", "data/processed", "db", "config"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ All dependencies loaded.")


✅ All dependencies loaded.


## 2. Enterprise Database Layer (SQLite — Operational Tables)



In [6]:
DB_PATH = "enterprise_intelligence.db"

def build_enterprise_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # ── Table 1: Startup Funding Records ──────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS startup_funding (
        id INTEGER PRIMARY KEY,
        company_name TEXT, sector TEXT, funding_round TEXT,
        amount_usd REAL, valuation_usd REAL, investor TEXT,
        country TEXT, year INTEGER, month INTEGER
    )
    """)
    funding_data = [
        ("NeuralEdge AI","AI/ML","Series B",42_000_000,210_000_000,"Sequoia Capital","USA",2024,3),
        ("QuantaHealth","HealthTech","Series A",18_500_000,74_000_000,"Andreessen Horowitz","USA",2024,5),
        ("GridFlow Energy","CleanTech","Seed",3_200_000,16_000_000,"Y Combinator","India",2024,1),
        ("LogiSense","Supply Chain","Series C",95_000_000,475_000_000,"Tiger Global","Singapore",2023,11),
        ("CodeForge","DevTools","Series A",22_000_000,110_000_000,"Accel","UK",2024,7),
        ("DataMesh Labs","Data Infra","Seed",5_500_000,27_500_000,"GV","Germany",2024,2),
        ("FinBridge","FinTech","Series B",67_000_000,335_000_000,"SoftBank","Brazil",2023,9),
        ("AgroVision","AgriTech","Series A",14_000_000,56_000_000,"Khosla Ventures","India",2024,4),
        ("SecureVault","Cybersecurity","Series C",120_000_000,600_000_000,"Coatue","USA",2024,6),
        ("EduPulse","EdTech","Seed",2_800_000,14_000_000,"500 Startups","Nigeria",2024,8),
        ("CloudNative Inc","Cloud","Series D",200_000_000,1_200_000_000,"Blackrock","USA",2023,12),
        ("MediScan AI","HealthTech","Series B",55_000_000,275_000_000,"NEA","USA",2024,9),
        ("RetailIQ","RetailTech","Series A",19_000_000,76_000_000,"Lightspeed","UK",2024,10),
        ("PayStream","FinTech","Seed",4_100_000,20_500_000,"Hustle Fund","Mexico",2024,1),
        ("RoboFlow","Robotics","Series B",38_000_000,190_000_000,"CRV","USA",2023,8),
    ]
    cur.executemany("""
        INSERT OR IGNORE INTO startup_funding
        (company_name,sector,funding_round,amount_usd,valuation_usd,investor,country,year,month)
        VALUES (?,?,?,?,?,?,?,?,?)
    """, funding_data)

    # ── Table 2: Sales Pipeline ───────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS sales_pipeline (
        id INTEGER PRIMARY KEY, deal_name TEXT, account TEXT, stage TEXT,
        deal_value REAL, probability REAL, owner TEXT, region TEXT,
        created_date TEXT, close_date TEXT, product TEXT
    )
    """)
    stages = ["Prospecting","Qualification","Proposal","Negotiation","Closed Won","Closed Lost"]
    owners = ["Alice Chen","Bob Patel","Carlos Ruiz","Diana Kim","Ethan Nair"]
    regions = ["APAC","EMEA","North America","LATAM"]
    products = ["Platform Pro","Analytics Suite","DataBridge","SecureOps","AI Copilot"]
    np.random.seed(42)
    pipeline_rows = []
    base_date = datetime(2024, 1, 1)
    for i in range(60):
        stage = np.random.choice(stages)
        prob = {"Prospecting":10,"Qualification":25,"Proposal":50,
                "Negotiation":75,"Closed Won":100,"Closed Lost":0}[stage]
        created = base_date + timedelta(days=int(np.random.randint(0, 300)))
        close = created + timedelta(days=int(np.random.randint(30, 180)))
        pipeline_rows.append((
            f"Deal-{i+1:03d}", f"Account-{np.random.randint(1,30):02d}", stage,
            round(np.random.uniform(10_000, 500_000), 2), prob,
            np.random.choice(owners), np.random.choice(regions),
            created.strftime("%Y-%m-%d"), close.strftime("%Y-%m-%d"),
            np.random.choice(products)
        ))
    cur.executemany("""
        INSERT OR IGNORE INTO sales_pipeline
        (deal_name,account,stage,deal_value,probability,owner,region,created_date,close_date,product)
        VALUES (?,?,?,?,?,?,?,?,?,?)
    """, pipeline_rows)

    # ── Table 3: Product KPIs ─────────────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS product_kpis (
        id INTEGER PRIMARY KEY, date TEXT, product TEXT,
        dau INTEGER, mau INTEGER, revenue REAL, churn_rate REAL,
        nps_score REAL, latency_p99_ms REAL, error_rate REAL
    )
    """)
    products_list = ["Platform Pro","Analytics Suite","DataBridge"]
    kpi_rows = []
    for prod in products_list:
        base_dau = {"Platform Pro":12000,"Analytics Suite":8500,"DataBridge":5200}[prod]
        base_rev = {"Platform Pro":85000,"Analytics Suite":62000,"DataBridge":41000}[prod]
        for week in range(24):
            date = (datetime(2024,1,1)+timedelta(weeks=week)).strftime("%Y-%m-%d")
            trend = 1 + 0.02*week
            noise = np.random.uniform(0.92, 1.08)
            kpi_rows.append((
                date, prod,
                int(base_dau*trend*noise), int(base_dau*trend*noise*4.2),
                round(base_rev*trend*noise, 2),
                round(np.random.uniform(1.2, 4.8), 2),
                round(np.random.uniform(38, 72), 1),
                round(np.random.uniform(120, 450), 1),
                round(np.random.uniform(0.1, 1.8), 3)
            ))
    cur.executemany("""
        INSERT OR IGNORE INTO product_kpis
        (date,product,dau,mau,revenue,churn_rate,nps_score,latency_p99_ms,error_rate)
        VALUES (?,?,?,?,?,?,?,?,?)
    """, kpi_rows)

    # ── Table 4: Operational Metrics ──────────────────────────────────────────
    cur.execute("""
    CREATE TABLE IF NOT EXISTS operational_metrics (
        id INTEGER PRIMARY KEY, timestamp TEXT, service TEXT,
        cpu_pct REAL, memory_pct REAL, request_count INTEGER,
        error_count INTEGER, avg_latency_ms REAL, region TEXT
    )
    """)
    services = ["API Gateway","ML Inference","Data Pipeline","Auth Service","Query Engine"]
    op_rows = []
    for i in range(120):
        ts = (datetime(2024,6,1)+timedelta(hours=i)).strftime("%Y-%m-%d %H:%M:%S")
        svc = np.random.choice(services)
        anomaly = np.random.random() < 0.08
        op_rows.append((
            ts, svc,
            round(np.random.uniform(60,95) if anomaly else np.random.uniform(20,65), 1),
            round(np.random.uniform(70,92) if anomaly else np.random.uniform(30,70), 1),
            int(np.random.randint(800, 5000)),
            int(np.random.randint(50,300) if anomaly else np.random.randint(0,20)),
            round(np.random.uniform(400,1200) if anomaly else np.random.uniform(50,250), 1),
            np.random.choice(regions)
        ))
    cur.executemany("""
        INSERT OR IGNORE INTO operational_metrics
        (timestamp,service,cpu_pct,memory_pct,request_count,error_count,avg_latency_ms,region)
        VALUES (?,?,?,?,?,?,?,?)
    """, op_rows)

    conn.commit()
    conn.close()
    print(f"✅ Enterprise DB built at '{DB_PATH}'")
    print("   Tables: startup_funding | sales_pipeline | product_kpis | operational_metrics")

build_enterprise_db()


✅ Enterprise DB built at 'enterprise_intelligence.db'
   Tables: startup_funding | sales_pipeline | product_kpis | operational_metrics


## 3. Crunchbase CSV Preprocessing Pipeline



In [7]:
RAW = Path("data/raw")
PROCESSED = Path("data/processed")

def categorize_role(title: str) -> str:
    """Derive role category from job title string."""
    t = str(title).lower()
    if any(x in t for x in ["ceo","founder","cto","coo","cfo","president"]):
        return "C-Suite/Founder"
    if any(x in t for x in ["vp","vice president","director"]):
        return "VP/Director"
    if "board" in t or "advisor" in t:
        return "Board/Advisor"
    if any(x in t for x in ["engineer","developer","architect"]):
        return "Engineering"
    return "Other"


def clean_objects(df: pd.DataFrame = None) -> pd.DataFrame:
    """Clean and normalize the objects (core entity) CSV."""
    if df is None:
        p = RAW / "objects.csv"
        if not p.exists():
            print("  ⚠️  objects.csv not found — generating synthetic Crunchbase data")
            return None
        df = pd.read_csv(p, low_memory=False)

    # Filter to usable entity types
    if "entity_type" in df.columns:
        df = df[df["entity_type"].isin(["Company","Person","FinancialOrg"])].copy()

    # Text normalization
    for col in ["name","category_code","status","country_code"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    if "normalized_name" not in df.columns and "name" in df.columns:
        df["normalized_name"] = df["name"].str.lower()
    if "category_code" in df.columns:
        df["category_code"] = df["category_code"].str.lower().replace("nan","unknown").fillna("unknown")
    if "status" in df.columns:
        df["status"] = df["status"].str.lower().replace("nan","unknown").fillna("unknown")
    if "country_code" in df.columns:
        df["country_code"] = df["country_code"].str.upper().replace("NAN","UNKNOWN").fillna("UNKNOWN")

    # Date parsing
    date_cols = ["founded_at","closed_at","first_funding_at","last_funding_at",
                 "first_milestone_at","last_milestone_at","created_at","updated_at"]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")

    # Numerics
    for col in ["funding_total_usd","investment_rounds","invested_companies",
                "funding_rounds","milestones","relationships"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Text truncation
    if "overview" in df.columns:
        df["overview"] = df["overview"].fillna("").astype(str).str[:2000]
    if "description" in df.columns:
        df["description"] = df["description"].fillna("").astype(str).str[:500]
    if "tag_list" in df.columns:
        df["tag_list"] = df["tag_list"].fillna("")

    # Drop image columns
    df = df.drop(columns=["logo_url","logo_width","logo_height",
                           "twitter_username","homepage_url"], errors="ignore")

    df.to_csv(PROCESSED / "objects_clean.csv", index=False)
    print(f"  objects: {len(df):,} rows cleaned")
    return df


def clean_acquisitions() -> pd.DataFrame:
    p = RAW / "acquisitions.csv"
    if not p.exists():
        print("  ⚠️  acquisitions.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    if "acquired_at" in df.columns:
        df["acquired_at"] = pd.to_datetime(df["acquired_at"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    if "price_amount" in df.columns:
        df["price_amount"] = pd.to_numeric(df["price_amount"], errors="coerce")
    if "price_currency_code" in df.columns:
        df["price_currency_code"] = df["price_currency_code"].fillna("USD")
    if "term_code" in df.columns:
        df["term_code"] = df["term_code"].fillna("unknown")
    df.to_csv(PROCESSED / "acquisitions_clean.csv", index=False)
    print(f"  acquisitions: {len(df):,} rows cleaned")
    return df


def clean_ipos() -> pd.DataFrame:
    p = RAW / "ipos.csv"
    if not p.exists():
        print("  ⚠️  ipos.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    if "public_at" in df.columns:
        df["public_at"] = pd.to_datetime(df["public_at"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    for col in ["valuation_amount","raised_amount"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df.to_csv(PROCESSED / "ipos_clean.csv", index=False)
    print(f"  ipos: {len(df):,} rows cleaned")
    return df


def clean_people() -> pd.DataFrame:
    p = RAW / "people.csv"
    if not p.exists():
        print("  ⚠️  people.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    if "first_name" in df.columns and "last_name" in df.columns:
        df["full_name"] = df["first_name"].fillna("") + " " + df["last_name"].fillna("")
        df["full_name"] = df["full_name"].str.strip()
    df.to_csv(PROCESSED / "people_clean.csv", index=False)
    print(f"  people: {len(df):,} rows cleaned")
    return df


def clean_relationships() -> pd.DataFrame:
    p = RAW / "relationships.csv"
    if not p.exists():
        print("  ⚠️  relationships.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    for col in ["start_at","end_at"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    if "title" in df.columns:
        df["title"] = df["title"].fillna("Unknown").str.strip()
        df["role_category"] = df["title"].apply(categorize_role)
    df.to_csv(PROCESSED / "relationships_clean.csv", index=False)
    print(f"  relationships: {len(df):,} rows cleaned")
    return df


def clean_offices() -> pd.DataFrame:
    p = RAW / "offices.csv"
    if not p.exists():
        print("  ⚠️  offices.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    if "country_code" in df.columns:
        df["country_code"] = df["country_code"].str.strip().str.upper().fillna("UNKNOWN")
    if "city" in df.columns:
        df["city"] = df["city"].str.strip().fillna("")
    for col in ["latitude","longitude"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df.to_csv(PROCESSED / "offices_clean.csv", index=False)
    print(f"  offices: {len(df):,} rows cleaned")
    return df


def clean_milestones() -> pd.DataFrame:
    p = RAW / "milestones.csv"
    if not p.exists():
        print("  ⚠️  milestones.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    if "milestone_at" in df.columns:
        df["milestone_at"] = pd.to_datetime(df["milestone_at"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    if "description" in df.columns:
        df["description"] = df["description"].fillna("").astype(str).str[:500]
    if "milestone_code" in df.columns:
        df["milestone_code"] = df["milestone_code"].fillna("other")
    df.to_csv(PROCESSED / "milestones_clean.csv", index=False)
    print(f"  milestones: {len(df):,} rows cleaned")
    return df


def clean_degrees() -> pd.DataFrame:
    p = RAW / "degrees.csv"
    if not p.exists():
        print("  ⚠️  degrees.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    for col in ["degree_type","subject","institution"]:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")
    if "graduated_at" in df.columns:
        df["graduated_at"] = pd.to_datetime(df["graduated_at"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    df.to_csv(PROCESSED / "degrees_clean.csv", index=False)
    print(f"  degrees: {len(df):,} rows cleaned")
    return df


def clean_funds() -> pd.DataFrame:
    p = RAW / "funds.csv"
    if not p.exists():
        print("  ⚠️  funds.csv not found — skipping")
        return None
    df = pd.read_csv(p, low_memory=False)
    if "funded_at" in df.columns:
        df["funded_at"] = pd.to_datetime(df["funded_at"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    if "raised_amount" in df.columns:
        df["raised_amount"] = pd.to_numeric(df["raised_amount"], errors="coerce")
    df.to_csv(PROCESSED / "funds_clean.csv", index=False)
    print(f"  funds: {len(df):,} rows cleaned")
    return df


print("🔧 Running Crunchbase preprocessing pipeline...")
clean_objects()
clean_acquisitions()
clean_ipos()
clean_people()
clean_relationships()
clean_offices()
clean_milestones()
clean_degrees()
clean_funds()
print("✅ Preprocessing complete.")


🔧 Running Crunchbase preprocessing pipeline...
  objects: 434,913 rows cleaned
  acquisitions: 9,562 rows cleaned
  ipos: 1,259 rows cleaned
  people: 226,709 rows cleaned
  relationships: 402,878 rows cleaned
  offices: 112,718 rows cleaned
  milestones: 39,456 rows cleaned
  degrees: 109,610 rows cleaned
  funds: 1,564 rows cleaned
✅ Preprocessing complete.


## 4. Crunchbase Synthetic Fallback Data

When real Crunchbase CSVs are not present, this cell generates a realistic synthetic
dataset that mirrors the exact schema structure — allowing all downstream agents,
SQL queries, and analytics to run end-to-end without the raw data files.

In [8]:
CB_DB_PATH = "db/crunchbase_ecosystem.db"

def generate_synthetic_crunchbase():
    """
    Generate synthetic Crunchbase-schema-compatible data for all 9 tables.
    Called automatically when real CSVs are missing.
    """
    np.random.seed(2024)
    sectors = ["web","mobile","enterprise","biotech","cleantech","fintech",
               "edtech","healthtech","ai","saas","hardware","media","ecommerce"]
    statuses = ["operating","acquired","closed","ipo"]
    countries = ["USA","GBR","DEU","IND","SGP","BRA","ISR","FRA","CAN","AUS"]
    role_cats = ["C-Suite/Founder","VP/Director","Board/Advisor","Engineering","Other"]

    n_companies = 2000
    n_people    = 500
    n_investors = 150

    # ── cb_objects: companies ─────────────────────────────────────────────────
    company_ids = [f"c:{i}" for i in range(1, n_companies+1)]
    company_names = [f"Company_{i}" for i in range(1, n_companies+1)]
    descriptions = [
        f"A {np.random.choice(sectors)} company providing innovative solutions in {np.random.choice(['B2B','B2C','enterprise','consumer'])} markets.",
        f"Leading provider of {np.random.choice(['cloud','AI','mobile','data'])} technology for modern businesses.",
        f"Disrupting the {np.random.choice(sectors)} industry with cutting-edge platform solutions.",
        f"Enterprise-grade software helping Fortune 500 companies optimize operations.",
        f"Next-generation {np.random.choice(['analytics','automation','collaboration'])} platform.",
    ]
    funded_years = np.random.randint(2000, 2022, n_companies)
    objects_rows = []
    for i, (cid, cname) in enumerate(zip(company_ids, company_names)):
        sector = np.random.choice(sectors)
        status = np.random.choice(statuses, p=[0.6, 0.2, 0.1, 0.1])
        funded_yr = funded_years[i]
        total_funding = np.random.lognormal(15, 2) if np.random.random() > 0.2 else 0
        rounds = int(np.random.randint(0, 8)) if total_funding > 0 else 0
        objects_rows.append((
            cid, "Company", i+1, None, cname, cname.lower(), cname.lower(),
            sector, status,
            f"{funded_yr}-{np.random.randint(1,13):02d}-01", None,
            f"{cname.lower()}.com",
            np.random.choice(descriptions),
            f"{np.random.choice(descriptions)} Overview text.",
            f"{sector},{np.random.choice(sectors)}",
            np.random.choice(countries), None, f"City_{i%50}", None,
            f"{funded_yr}-{np.random.randint(1,13):02d}-01" if total_funding > 0 else None,
            f"{funded_yr+rounds}-{np.random.randint(1,13):02d}-01" if total_funding > 0 else None,
            rounds, round(total_funding, 2), None, None, 0, 0,
            "2023-01-01", "2024-01-01"
        ))

    # ── cb_objects: financial orgs (investors) ────────────────────────────────
    investor_ids = [f"f:{i}" for i in range(1, n_investors+1)]
    investor_names = [
        "Sequoia Capital","Andreessen Horowitz","Kleiner Perkins","Benchmark Capital",
        "Accel Partners","GV","Bessemer Venture Partners","Lightspeed",
        "Tiger Global","SoftBank Vision Fund","Coatue Management","CRV",
        "NEA","Khosla Ventures","Founders Fund","Union Square Ventures",
        "Index Ventures","Insight Partners","General Atlantic","Warburg Pincus"
    ] + [f"VentureCapital_{i}" for i in range(21, n_investors+1)]
    for i, (fid, fname) in enumerate(zip(investor_ids, investor_names)):
        objects_rows.append((
            fid, "FinancialOrg", n_companies+i+1, None, fname, fname.lower(), fname.lower(),
            "finance", "operating", None, None, f"{fname.lower().replace(' ','-')}.com",
            f"Leading venture capital firm investing in {np.random.choice(sectors)} startups.",
            "Investment firm overview.", "venture capital,finance",
            np.random.choice(countries[:5]), None, "San Francisco", None,
            None, None, 0, 0, None, None, 0, 0, "2020-01-01", "2024-01-01"
        ))

    # ── cb_objects: people ────────────────────────────────────────────────────
    person_ids = [f"p:{i}" for i in range(1, n_people+1)]
    first_names = ["James","Maria","David","Sarah","Michael","Lisa","John","Emily",
                   "Robert","Anna","William","Jessica","Richard","Amanda","Joseph",
                   "Stephanie","Thomas","Jennifer","Charles","Linda"]
    last_names = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller",
                  "Davis","Wilson","Anderson","Taylor","Thomas","Jackson","White","Harris"]
    for i, pid in enumerate(person_ids):
        fn = np.random.choice(first_names)
        ln = np.random.choice(last_names)
        objects_rows.append((
            pid, "Person", n_companies+n_investors+i+1, None, f"{fn} {ln}",
            f"{fn} {ln}".lower(), f"{fn.lower()}.{ln.lower()}", "person", "operating",
            None, None, None, f"Executive with {np.random.randint(5,25)} years experience.",
            None, None, np.random.choice(countries), None, "Various", None,
            None, None, 0, 0, None, None, 0, 0, "2020-01-01", "2024-01-01"
        ))

    # ── Persist cb_objects ─────────────────────────────────────────────────────
    obj_cols = [
        "id","entity_type","entity_id","parent_id","name","normalized_name","permalink",
        "category_code","status","founded_at","closed_at","domain","description","overview",
        "tag_list","country_code","state_code","city","region",
        "first_funding_at","last_funding_at","funding_rounds","funding_total_usd",
        "first_milestone_at","last_milestone_at","milestones","relationships",
        "created_at","updated_at"
    ]
    df_obj = pd.DataFrame(objects_rows, columns=obj_cols)
    df_obj.to_csv(PROCESSED / "objects_clean.csv", index=False)

    # ── cb_acquisitions ───────────────────────────────────────────────────────
    n_acq = 300
    acq_rows = []
    for i in range(n_acq):
        acq_rows.append((
            i+1, i+1,
            np.random.choice(investor_ids + company_ids[:50]),
            np.random.choice(company_ids),
            np.random.choice(["cash","stock","unknown"]),
            np.random.lognormal(18, 2) if np.random.random() > 0.3 else None,
            "USD",
            f"{np.random.randint(2000,2023)}-{np.random.randint(1,13):02d}-01",
            f"Acquisition {i+1}",
            "2023-01-01"
        ))
    df_acq = pd.DataFrame(acq_rows, columns=[
        "id","acquisition_id","acquiring_object_id","acquired_object_id",
        "term_code","price_amount","price_currency_code","acquired_at",
        "source_description","created_at"
    ])
    df_acq.to_csv(PROCESSED / "acquisitions_clean.csv", index=False)

    # ── cb_ipos ───────────────────────────────────────────────────────────────
    ipo_companies = np.random.choice(company_ids, size=80, replace=False)
    ipo_rows = [(
        i+1, i+1, cid,
        np.random.lognormal(20, 1.5),
        "USD",
        np.random.lognormal(19, 1.5),
        "USD",
        f"{np.random.randint(1998,2023)}-{np.random.randint(1,13):02d}-01",
        f"TICK{i:03d}",
        "2023-01-01"
    ) for i, cid in enumerate(ipo_companies)]
    df_ipos = pd.DataFrame(ipo_rows, columns=[
        "id","ipo_id","object_id","valuation_amount","valuation_currency_code",
        "raised_amount","raised_currency_code","public_at","stock_symbol","created_at"
    ])
    df_ipos.to_csv(PROCESSED / "ipos_clean.csv", index=False)

    # ── cb_people ─────────────────────────────────────────────────────────────
    people_rows = [(
        i+1, pid,
        np.random.choice(first_names), np.random.choice(last_names),
        f"{np.random.choice(first_names)} {np.random.choice(last_names)}",
        np.random.choice(countries), np.random.choice(investor_names[:20])
    ) for i, pid in enumerate(person_ids)]
    df_people = pd.DataFrame(people_rows, columns=[
        "id","object_id","first_name","last_name","full_name","birthplace","affiliation_name"
    ])
    df_people.to_csv(PROCESSED / "people_clean.csv", index=False)

    # ── cb_relationships ──────────────────────────────────────────────────────
    rel_rows = [(
        i+1, i+1,
        np.random.choice(person_ids),
        np.random.choice(company_ids),
        f"{np.random.randint(2000,2020)}-01-01",
        None if np.random.random() > 0.3 else f"{np.random.randint(2021,2024)}-01-01",
        0 if np.random.random() > 0.3 else 1,
        np.random.choice(["CEO","CTO","VP Engineering","Founder","Director","Board Member",
                          "Software Engineer","Product Manager"]),
        np.random.choice(role_cats),
        "2023-01-01"
    ) for i in range(1500)]
    df_rel = pd.DataFrame(rel_rows, columns=[
        "id","relationship_id","person_object_id","relationship_object_id",
        "start_at","end_at","is_past","title","role_category","created_at"
    ])
    df_rel.to_csv(PROCESSED / "relationships_clean.csv", index=False)

    # ── cb_offices ────────────────────────────────────────────────────────────
    offices_rows = [(
        i+1,
        np.random.choice(company_ids + investor_ids),
        i+1, f"HQ Office {i+1}", None, f"123 Main St Suite {i+1}",
        f"City_{i%30}", f"{np.random.randint(10000,99999)}",
        None, np.random.choice(countries),
        round(np.random.uniform(-90, 90), 4),
        round(np.random.uniform(-180, 180), 4)
    ) for i in range(600)]
    df_off = pd.DataFrame(offices_rows, columns=[
        "id","object_id","office_id","description","region","address1",
        "city","zip_code","state_code","country_code","latitude","longitude"
    ])
    df_off.to_csv(PROCESSED / "offices_clean.csv", index=False)

    # ── cb_milestones ─────────────────────────────────────────────────────────
    milestone_descs = [
        "Launched new product line","Reached 1M users","Secured Series A funding",
        "Opened APAC office","Won industry award","Partnership with enterprise client",
        "Launched mobile app","Achieved SOC2 certification","Acquired competitor",
        "Crossed $10M ARR","Reached profitability","Launched in EU market"
    ]
    ms_rows = [(
        i+1,
        np.random.choice(company_ids),
        f"{np.random.randint(2010,2023)}-{np.random.randint(1,13):02d}-01",
        np.random.choice(["product_launch","funding","partnership","award","expansion","other"]),
        np.random.choice(milestone_descs),
        None, None, "2023-01-01"
    ) for i in range(800)]
    df_ms = pd.DataFrame(ms_rows, columns=[
        "id","object_id","milestone_at","milestone_code","description",
        "source_url","source_description","created_at"
    ])
    df_ms.to_csv(PROCESSED / "milestones_clean.csv", index=False)

    # ── cb_degrees ────────────────────────────────────────────────────────────
    degree_rows = [(
        i+1, np.random.choice(person_ids),
        np.random.choice(["BS","MS","MBA","PhD","BA"]),
        np.random.choice(["Computer Science","Business","Engineering","Mathematics","Physics"]),
        np.random.choice(["MIT","Stanford","Harvard","Carnegie Mellon","UC Berkeley",
                          "Oxford","Cambridge","ETH Zurich","Caltech","Princeton"]),
        f"{np.random.randint(1990,2018)}-06-01", "2023-01-01"
    ) for i in range(400)]
    df_deg = pd.DataFrame(degree_rows, columns=[
        "id","object_id","degree_type","subject","institution","graduated_at","created_at"
    ])
    df_deg.to_csv(PROCESSED / "degrees_clean.csv", index=False)

    # ── cb_funds ──────────────────────────────────────────────────────────────
    fund_rows = [(
        i+1, i+1,
        np.random.choice(investor_ids),
        f"Fund {np.random.choice(['I','II','III','IV','V','VI'])} - {np.random.randint(2005,2023)}",
        f"{np.random.randint(2000,2023)}-{np.random.randint(1,13):02d}-01",
        np.random.lognormal(20, 1.5),
        "USD", f"Venture fund raised by {np.random.choice(investor_names[:20])}", "2023-01-01"
    ) for i in range(200)]
    df_funds = pd.DataFrame(fund_rows, columns=[
        "id","fund_id","object_id","name","funded_at","raised_amount",
        "raised_currency_code","source_description","created_at"
    ])
    df_funds.to_csv(PROCESSED / "funds_clean.csv", index=False)

    print("✅ Synthetic Crunchbase data generated:")
    print(f"   objects: {len(df_obj):,} | acquisitions: {len(df_acq):,} | ipos: {len(df_ipos):,}")
    print(f"   people: {len(df_people):,} | relationships: {len(df_rel):,} | offices: {len(df_off):,}")
    print(f"   milestones: {len(df_ms):,} | degrees: {len(df_deg):,} | funds: {len(df_funds):,}")
    return True


# Only generate synthetic data if real CSVs are missing
missing = not (PROCESSED / "objects_clean.csv").exists()
if missing:
    print("🔧 Real CSVs not found. Generating synthetic Crunchbase data...")
    generate_synthetic_crunchbase()
else:
    print("✅ Cleaned CSVs already present in data/processed/")


✅ Cleaned CSVs already present in data/processed/


## 5. Crunchbase Relational Database Assembly

Builds the 9-table relational schema in a separate `crunchbase_ecosystem.db`,
with FK constraints, WAL journaling, and 18 performance indexes.
Loads cleaned CSVs in 50k-row chunks.

In [9]:
def build_crunchbase_schema(conn: sqlite3.Connection):
    """Create all 9 Crunchbase tables plus placeholder tables for future data."""
    cur = conn.cursor()
    cur.executescript("""
    PRAGMA foreign_keys = ON;
    PRAGMA journal_mode = WAL;
    PRAGMA synchronous = NORMAL;

    CREATE TABLE IF NOT EXISTS cb_objects (
        id TEXT PRIMARY KEY, entity_type TEXT NOT NULL, entity_id INTEGER,
        parent_id TEXT, name TEXT, normalized_name TEXT, permalink TEXT,
        category_code TEXT, status TEXT, founded_at TEXT, closed_at TEXT,
        domain TEXT, description TEXT, overview TEXT, tag_list TEXT,
        country_code TEXT, state_code TEXT, city TEXT, region TEXT,
        first_funding_at TEXT, last_funding_at TEXT,
        funding_rounds INTEGER DEFAULT 0, funding_total_usd REAL DEFAULT 0,
        first_milestone_at TEXT, last_milestone_at TEXT,
        milestones INTEGER DEFAULT 0, relationships INTEGER DEFAULT 0,
        created_at TEXT, updated_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_acquisitions (
        id INTEGER PRIMARY KEY, acquisition_id INTEGER UNIQUE,
        acquiring_object_id TEXT, acquired_object_id TEXT,
        term_code TEXT, price_amount REAL, price_currency_code TEXT DEFAULT 'USD',
        acquired_at TEXT, source_description TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_ipos (
        id INTEGER PRIMARY KEY, ipo_id INTEGER UNIQUE,
        object_id TEXT, valuation_amount REAL, valuation_currency_code TEXT DEFAULT 'USD',
        raised_amount REAL, raised_currency_code TEXT DEFAULT 'USD',
        public_at TEXT, stock_symbol TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_people (
        id INTEGER PRIMARY KEY, object_id TEXT UNIQUE,
        first_name TEXT, last_name TEXT, full_name TEXT,
        birthplace TEXT, affiliation_name TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_relationships (
        id INTEGER PRIMARY KEY, relationship_id INTEGER UNIQUE,
        person_object_id TEXT, relationship_object_id TEXT,
        start_at TEXT, end_at TEXT, is_past INTEGER DEFAULT 0,
        title TEXT, role_category TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_offices (
        id INTEGER PRIMARY KEY, object_id TEXT, office_id INTEGER,
        description TEXT, region TEXT, address1 TEXT, city TEXT,
        zip_code TEXT, state_code TEXT, country_code TEXT,
        latitude REAL, longitude REAL
    );

    CREATE TABLE IF NOT EXISTS cb_milestones (
        id INTEGER PRIMARY KEY, object_id TEXT, milestone_at TEXT,
        milestone_code TEXT, description TEXT, source_url TEXT,
        source_description TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_degrees (
        id INTEGER PRIMARY KEY, object_id TEXT,
        degree_type TEXT, subject TEXT, institution TEXT,
        graduated_at TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_funds (
        id INTEGER PRIMARY KEY, fund_id INTEGER UNIQUE, object_id TEXT,
        name TEXT, funded_at TEXT, raised_amount REAL,
        raised_currency_code TEXT DEFAULT 'USD',
        source_description TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_funding_rounds (
        id INTEGER PRIMARY KEY, funding_round_id INTEGER UNIQUE,
        object_id TEXT, funded_at TEXT, funding_round_type TEXT,
        raised_amount_usd REAL, pre_money_valuation REAL,
        participants INTEGER, is_first_round INTEGER DEFAULT 0,
        is_last_round INTEGER DEFAULT 0, source_description TEXT, created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS cb_investments (
        id INTEGER PRIMARY KEY, funding_round_id INTEGER,
        funded_object_id TEXT, investor_object_id TEXT,
        investor_name TEXT, is_lead_investor INTEGER DEFAULT 0, created_at TEXT
    );
    """)
    conn.commit()


def build_indexes(conn: sqlite3.Connection):
    """Build 18 performance indexes across all Crunchbase tables."""
    cur = conn.cursor()
    indexes = [
        "CREATE INDEX IF NOT EXISTS idx_obj_entity_type ON cb_objects(entity_type)",
        "CREATE INDEX IF NOT EXISTS idx_obj_category    ON cb_objects(category_code)",
        "CREATE INDEX IF NOT EXISTS idx_obj_status      ON cb_objects(status)",
        "CREATE INDEX IF NOT EXISTS idx_obj_country     ON cb_objects(country_code)",
        "CREATE INDEX IF NOT EXISTS idx_obj_founded     ON cb_objects(founded_at)",
        "CREATE INDEX IF NOT EXISTS idx_obj_funding     ON cb_objects(funding_total_usd)",
        "CREATE INDEX IF NOT EXISTS idx_acq_acquirer    ON cb_acquisitions(acquiring_object_id)",
        "CREATE INDEX IF NOT EXISTS idx_acq_acquired    ON cb_acquisitions(acquired_object_id)",
        "CREATE INDEX IF NOT EXISTS idx_acq_date        ON cb_acquisitions(acquired_at)",
        "CREATE INDEX IF NOT EXISTS idx_ipo_obj         ON cb_ipos(object_id)",
        "CREATE INDEX IF NOT EXISTS idx_ipo_date        ON cb_ipos(public_at)",
        "CREATE INDEX IF NOT EXISTS idx_rel_person      ON cb_relationships(person_object_id)",
        "CREATE INDEX IF NOT EXISTS idx_rel_company     ON cb_relationships(relationship_object_id)",
        "CREATE INDEX IF NOT EXISTS idx_rel_role        ON cb_relationships(role_category)",
        "CREATE INDEX IF NOT EXISTS idx_off_obj         ON cb_offices(object_id)",
        "CREATE INDEX IF NOT EXISTS idx_off_country     ON cb_offices(country_code)",
        "CREATE INDEX IF NOT EXISTS idx_mil_obj         ON cb_milestones(object_id)",
        "CREATE INDEX IF NOT EXISTS idx_funds_obj       ON cb_funds(object_id)",
    ]
    for idx in indexes:
        cur.execute(idx)
    conn.commit()
    print(f"  ✅ {len(indexes)} indexes created")


def load_csv_chunked(conn, csv_path: Path, table_name: str, chunk_size: int = 50_000):
    """Load a cleaned CSV into SQLite in chunks (handles 1M+ rows)."""
    total = 0
    try:
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size, low_memory=False):
            # Drop columns that don't exist in target table
            chunk.to_sql(table_name, conn, if_exists="append", index=False, method="multi")
            total += len(chunk)
        print(f"  Loaded {total:,} rows → {table_name}")
    except Exception as e:
        print(f"  ⚠️  Error loading {table_name}: {e}")
    return total


def build_crunchbase_db():
    """Assemble the complete Crunchbase relational database."""
    conn = sqlite3.connect(CB_DB_PATH)

    print("Building schema...")
    build_crunchbase_schema(conn)

    print("Loading tables...")
    table_map = {
        "objects_clean.csv":       "cb_objects",
        "acquisitions_clean.csv":  "cb_acquisitions",
        "ipos_clean.csv":          "cb_ipos",
        "people_clean.csv":        "cb_people",
        "relationships_clean.csv": "cb_relationships",
        "offices_clean.csv":       "cb_offices",
        "milestones_clean.csv":    "cb_milestones",
        "degrees_clean.csv":       "cb_degrees",
        "funds_clean.csv":         "cb_funds",
    }
    for csv_name, table_name in table_map.items():
        path = PROCESSED / csv_name
        if path.exists():
            load_csv_chunked(conn, path, table_name)
        else:
            print(f"  ⚠️  {csv_name} not found — skipping {table_name}")

    print("Building indexes...")
    build_indexes(conn)
    conn.close()
    print(f"✅ Crunchbase DB assembled at {CB_DB_PATH}")


build_crunchbase_db()


Building schema...
Loading tables...
  ⚠️  Error loading cb_objects: too many SQL variables
  ⚠️  Error loading cb_acquisitions: table cb_acquisitions has no column named source_url
  ⚠️  Error loading cb_ipos: table cb_ipos has no column named source_url
  ⚠️  Error loading cb_people: too many SQL variables
  ⚠️  Error loading cb_relationships: too many SQL variables
  ⚠️  Error loading cb_offices: too many SQL variables
  ⚠️  Error loading cb_milestones: too many SQL variables
  ⚠️  Error loading cb_degrees: too many SQL variables
  ⚠️  Error loading cb_funds: table cb_funds has no column named source_url
Building indexes...
  ✅ 18 indexes created
✅ Crunchbase DB assembled at db/crunchbase_ecosystem.db


## 6. Schema Registry (Extended — Both Databases)

Generates schema strings for both databases with join hints for the SQL agent.
Also persists a machine-readable `schema_registry.json` for tooling.

In [10]:
CB_RELATIONSHIP_MAP = """
RELATIONAL JOIN MAP — use these patterns for multi-table Crunchbase queries:

1. Company + acquisition details:
   cb_objects o JOIN cb_acquisitions a ON o.id = a.acquired_object_id
   cb_objects acquirer JOIN cb_acquisitions a ON acquirer.id = a.acquiring_object_id

2. Company + IPO details:
   cb_objects o JOIN cb_ipos i ON o.id = i.object_id

3. Company founders/executives:
   cb_objects o
   JOIN cb_relationships r ON o.id = r.relationship_object_id
   JOIN cb_people p ON r.person_object_id = p.object_id

4. Investor funds raised:
   cb_objects o JOIN cb_funds f ON o.id = f.object_id
   WHERE o.entity_type = 'FinancialOrg'

5. Company office locations:
   cb_objects o JOIN cb_offices off ON o.id = off.object_id

6. Company milestones timeline:
   cb_objects o JOIN cb_milestones m ON o.id = m.object_id

7. People education background:
   cb_people p JOIN cb_degrees d ON p.object_id = d.object_id

KEY COLUMNS:
- cb_objects.id           → e.g. "c:1", "f:371", "p:2"
- cb_objects.entity_type  → "Company" | "FinancialOrg" | "Person"
- cb_objects.category_code→ "web" | "mobile" | "enterprise" | "biotech" | "ai" | "fintech" ...
- cb_objects.status       → "operating" | "acquired" | "closed" | "ipo"
- cb_acquisitions.price_amount → deal value in USD
- cb_ipos.valuation_amount → IPO valuation in USD
- cb_relationships.role_category → "C-Suite/Founder" | "VP/Director" | "Board/Advisor" | "Engineering" | "Other"
- cb_funds.raised_amount  → fund size in USD
"""


def get_db_schema() -> str:
    """Return schema string for the enterprise operational DB."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]
    parts = []
    for table in tables:
        cur.execute(f"PRAGMA table_info({table})")
        cols = cur.fetchall()
        col_defs = ", ".join(f"{c[1]} ({c[2]})" for c in cols)
        cur.execute(f"SELECT * FROM {table} LIMIT 2")
        sample = cur.fetchall()
        parts.append(f"TABLE: {table}\n  COLUMNS: {col_defs}\n  SAMPLE: {sample}")
    conn.close()
    return "\n\n".join(parts)


def get_cb_schema() -> str:
    """Return schema string for Crunchbase DB with join hints."""
    conn = sqlite3.connect(CB_DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]
    parts = [CB_RELATIONSHIP_MAP]
    for table in tables:
        cur.execute(f"PRAGMA table_info({table})")
        cols = cur.fetchall()
        col_defs = ", ".join(f"{c[1]} ({c[2]})" for c in cols)
        cur.execute(f"SELECT * FROM {table} LIMIT 1")
        sample = cur.fetchall()
        parts.append(f"TABLE: {table}\n  COLUMNS: {col_defs}\n  SAMPLE: {sample}")
    conn.close()
    return "\n\n".join(parts)


# Save machine-readable registry
schema_registry = {
    "databases": {
        "enterprise_intelligence": {
            "path": DB_PATH,
            "tables": ["startup_funding","sales_pipeline","product_kpis","operational_metrics"],
            "primary_use": "operational analytics, KPI tracking"
        },
        "crunchbase_ecosystem": {
            "path": CB_DB_PATH,
            "tables": ["cb_objects","cb_acquisitions","cb_ipos","cb_people",
                       "cb_relationships","cb_offices","cb_milestones","cb_degrees","cb_funds"],
            "primary_use": "startup ecosystem intelligence, investor analysis, M&A"
        }
    },
    "common_joins": [
        {"name":"company_acquisitions",
         "sql":"cb_objects o JOIN cb_acquisitions a ON o.id = a.acquired_object_id"},
        {"name":"company_ipo",
         "sql":"cb_objects o JOIN cb_ipos i ON o.id = i.object_id"},
        {"name":"company_founders",
         "sql":"cb_objects o JOIN cb_relationships r ON o.id = r.relationship_object_id JOIN cb_people p ON p.object_id = r.person_object_id WHERE r.role_category = 'C-Suite/Founder'"},
        {"name":"investor_funds",
         "sql":"cb_objects o JOIN cb_funds f ON o.id = f.object_id WHERE o.entity_type = 'FinancialOrg'"}
    ]
}
with open("config/schema_registry.json","w") as f:
    json.dump(schema_registry, f, indent=2)

DB_SCHEMA = get_db_schema()
CB_SCHEMA = get_cb_schema()
print("✅ Schema registry loaded.")
print(f"   Enterprise tables: {len(DB_SCHEMA.split('TABLE:'))-1}")
print(f"   Crunchbase tables: {len(CB_SCHEMA.split('TABLE:'))-1}")
print("   schema_registry.json saved to config/")


✅ Schema registry loaded.
   Enterprise tables: 4
   Crunchbase tables: 11
   schema_registry.json saved to config/


## 7. Hybrid Retrieval Engine (FAISS + BM25 + RRF)

Extends the original 10-document knowledge base with startup narratives and
investor profiles derived directly from the Crunchbase data.
- **FAISS**: dense semantic retrieval
- **BM25**: sparse keyword retrieval
- **RRF**: Reciprocal Rank Fusion merging both ranked lists

In [11]:
# ── Original enterprise knowledge base ────────────────────────────────────────
ENTERPRISE_DOCS = [
    """Enterprise AI adoption report 2024: 78% of Fortune 500 companies have deployed at least one
    production ML model. Key drivers include cost reduction (42%), operational efficiency (38%), and
    new revenue streams (20%). Industries leading adoption: FinTech, HealthTech, and Manufacturing.""",

    """RAG (Retrieval-Augmented Generation) architecture best practices: Use hybrid retrieval combining
    dense vector search with BM25 sparse retrieval. Apply Reciprocal Rank Fusion to merge ranked lists.
    Chunk size of 512 tokens with 10% overlap works well for technical documents. Evaluate with RAGAS.""",

    """LangGraph multi-agent orchestration: StateGraph nodes communicate through typed state objects.
    Use conditional edges for intent routing. ToolNode handles tool execution. MemorySaver provides
    cross-turn memory. Checkpointing enables fault tolerance in production deployments.""",

    """Startup funding landscape Q3 2024: AI/ML sector raised $4.2B in Q3, up 34% YoY.
    Series B rounds dominate by volume. APAC sees 28% growth driven by India and Southeast Asia.
    HealthTech and CleanTech remain investor favorites alongside enterprise SaaS.""",

    """Anomaly detection in operational metrics: Z-score method flags values beyond 2.5 standard
    deviations. IQR method robust to non-Gaussian distributions. For time-series, seasonal
    decomposition before z-scoring reduces false positives significantly.""",

    """Text-to-SQL agent design: Schema-aware prompting reduces hallucinated column names by 60%.
    Always include sample rows in the schema context. Use chain-of-thought before generating SQL.
    Validate generated SQL with EXPLAIN before execution. Multi-table queries require explicit JOIN hints.""",

    """Product KPI frameworks: DAU/MAU ratio above 20% signals strong engagement. Churn rate below
    2% monthly is healthy for enterprise SaaS. NPS above 50 is excellent. P99 latency should stay
    under 500ms for interactive products. Error rate above 1% triggers SLA review.""",

    """Observability in ML systems: Track token latency, retrieval latency, SQL execution time,
    and confidence scores per request. Store telemetry in structured logs. Build dashboards showing
    route distribution, p95/p99 latencies, and error rates.""",

    """Reinforcement Learning from Human Feedback (RLHF): Core technique for aligning LLMs.
    Phases: supervised fine-tuning, reward model training, and PPO optimization. Constitutional AI
    (CAI) is an alternative using AI feedback. DPO avoids the reward model entirely.""",

    """Sales pipeline analytics: Win rate by stage shows qualification effectiveness. Average deal
    size by region identifies geographic opportunities. Pipeline velocity is the key metric for
    revenue forecasting. CRM data quality directly impacts forecast accuracy.""",
]


def build_startup_corpus(limit: int = 3000) -> list:
    """Build startup narrative documents from Crunchbase data for RAG indexing."""
    conn = sqlite3.connect(CB_DB_PATH)
    companies = pd.read_sql_query(f"""
        SELECT o.id, o.name, o.category_code, o.status, o.country_code,
               o.founded_at, o.description, o.tag_list, o.funding_total_usd, o.funding_rounds
        FROM cb_objects o
        WHERE o.entity_type = 'Company'
          AND o.description != '' AND o.description IS NOT NULL
        LIMIT {limit}
    """, conn)
    milestones = pd.read_sql_query("""
        SELECT object_id, GROUP_CONCAT(description, '; ') as milestone_text
        FROM cb_milestones WHERE description != ''
        GROUP BY object_id
    """, conn)
    conn.close()

    companies = companies.merge(milestones, left_on="id", right_on="object_id", how="left")
    docs = []
    for _, row in companies.iterrows():
        text = (
            f"Company: {row['name']} | Sector: {row['category_code']} | "
            f"Status: {row['status']} | Country: {row['country_code']} | "
            f"Founded: {row['founded_at']} | "
            f"Funding: ${row['funding_total_usd']:,.0f} over {row['funding_rounds']:.0f} rounds. "
            f"Description: {row['description']} "
        )
        if pd.notna(row.get("milestone_text")):
            text += f"Milestones: {str(row['milestone_text'])[:300]}"
        if pd.notna(row.get("tag_list")) and row["tag_list"]:
            text += f" Tags: {row['tag_list']}"
        docs.append(text)
    return docs


def build_investor_corpus() -> list:
    """Build investor/VC narrative documents."""
    conn = sqlite3.connect(CB_DB_PATH)
    investors = pd.read_sql_query("""
        SELECT o.name, o.description, f.name as fund_name,
               f.raised_amount, f.funded_at
        FROM cb_objects o
        LEFT JOIN cb_funds f ON o.id = f.object_id
        WHERE o.entity_type = 'FinancialOrg'
          AND o.description IS NOT NULL AND o.description != ''
        LIMIT 400
    """, conn)
    conn.close()

    docs = []
    for name, group in investors.groupby("name"):
        r = group.iloc[0]
        text = f"Investor: {r['name']} | Type: FinancialOrg. Description: {r['description']} "
        fund_rows = group.dropna(subset=["fund_name"])
        if len(fund_rows) > 0:
            fund_names = ", ".join(fund_rows["fund_name"].tolist()[:3])
            total = fund_rows["raised_amount"].sum()
            text += f"Funds: {fund_names}. Total raised: ${total:,.0f}."
        docs.append(text)
    return docs


# ── Build extended corpus ──────────────────────────────────────────────────────
print("🔧 Building startup + investor corpus from Crunchbase data...")
startup_corpus  = build_startup_corpus(limit=3000)
investor_corpus = build_investor_corpus()
ENTERPRISE_DOCS_EXTENDED = ENTERPRISE_DOCS + startup_corpus + investor_corpus
print(f"  Corpus: {len(ENTERPRISE_DOCS)} enterprise + {len(startup_corpus)} startup + {len(investor_corpus)} investor = {len(ENTERPRISE_DOCS_EXTENDED)} total")

# ── FAISS + BM25 index ────────────────────────────────────────────────────────
print("🔧 Building FAISS + BM25 index...")
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

doc_objects = [
    Document(
        page_content=d,
        metadata={
            "source": ("crunchbase_startups" if i >= len(ENTERPRISE_DOCS) and i < len(ENTERPRISE_DOCS)+len(startup_corpus)
                       else "crunchbase_investors" if i >= len(ENTERPRISE_DOCS)+len(startup_corpus)
                       else f"enterprise_kb_{i}"),
            "doc_id": i,
            "type": ("startup" if i >= len(ENTERPRISE_DOCS) and i < len(ENTERPRISE_DOCS)+len(startup_corpus)
                     else "investor" if i >= len(ENTERPRISE_DOCS)+len(startup_corpus)
                     else "enterprise")
        }
    )
    for i, d in enumerate(ENTERPRISE_DOCS_EXTENDED)
]
chunks = splitter.split_documents(doc_objects)
faiss_store = FAISS.from_documents(chunks, embeddings_model)
faiss_retriever = faiss_store.as_retriever(search_type="similarity", search_kwargs={"k": 6})

tokenized_corpus = [doc.page_content.lower().split() for doc in chunks]
bm25_index = BM25Okapi(tokenized_corpus)
print(f"✅ Extended hybrid index ready: {len(chunks)} chunks (FAISS + BM25)")


🔧 Building startup + investor corpus from Crunchbase data...
  Corpus: 10 enterprise + 0 startup + 0 investor = 10 total
🔧 Building FAISS + BM25 index...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Extended hybrid index ready: 11 chunks (FAISS + BM25)


In [12]:
# ── Reciprocal Rank Fusion ─────────────────────────────────────────────────────

def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> list:
    """
    Merge multiple ranked doc lists using RRF.
    RRF score = sum(1 / (k + rank_i)) across all lists.
    k=60 smoothing constant reduces sensitivity to exact rank.
    """
    scores = defaultdict(float)
    doc_map = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked):
            key = doc.page_content[:80]
            scores[key] += 1.0 / (k + rank + 1)
            doc_map[key] = doc
    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
    return [doc_map[k] for k in sorted_keys]


def hybrid_retrieve(query: str, top_k: int = 4) -> list:
    """Combine FAISS (dense) + BM25 (sparse) results via RRF."""
    dense_results = faiss_retriever.invoke(query)
    tokens = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:8]
    sparse_results = [chunks[i] for i in top_indices if bm25_scores[i] > 0]
    fused = reciprocal_rank_fusion([dense_results, sparse_results])
    return fused[:top_k]

test_r = hybrid_retrieve("venture capital fund investment")
print("✅ Hybrid RRF retrieval verified.")
print(f"   Top result type: {test_r[0].metadata.get('type','?')} | snippet: {test_r[0].page_content[:80]}...")


✅ Hybrid RRF retrieval verified.
   Top result type: enterprise | snippet: Startup funding landscape Q3 2024: AI/ML sector raised $4.2B in Q3, up 34% YoY.
...


## 8. Observability & Telemetry Layer

Extended `TelemetryRecord` now tracks SQL retry counts, agent type,
and which database was queried — giving full production observability.

In [13]:
@dataclass
class TelemetryRecord:
    request_id: str
    timestamp: str
    query: str
    route: str                    # analytics|rag|forecasting|anomaly|investor|ecosystem|general
    total_latency_ms: float
    retrieval_latency_ms: float
    sql_latency_ms: float
    llm_latency_ms: float
    estimated_tokens: int
    confidence_score: float       # 0.0–1.0
    sources_cited: int
    validator_passed: bool
    sql_retry_count: int = 0      # NEW — tracks correction loop frequency
    agent_type: str = "standard"  # NEW — "standard"|"investor"|"ecosystem"|"forecasting"|"anomaly"
    db_queried: str = "enterprise"# NEW — "enterprise"|"crunchbase"|"both"
    error: Optional[str] = None

telemetry_log: list = []

def log_telemetry(record: TelemetryRecord):
    telemetry_log.append(record)

def get_telemetry_df() -> pd.DataFrame:
    if not telemetry_log:
        return pd.DataFrame()
    return pd.DataFrame([asdict(r) for r in telemetry_log])

print("✅ Extended telemetry layer initialized.")


✅ Extended telemetry layer initialized.


## 9. Enterprise State Schema (Extended)

In [14]:
class EnterpriseState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    route: Optional[str]           # analytics|rag|forecasting|anomaly|investor|ecosystem|general
    request_id: Optional[str]
    t_start: Optional[float]
    retrieval_ms: Optional[float]
    sql_ms: Optional[float]
    validator_passed: Optional[bool]
    confidence_score: Optional[float]
    sources: Optional[list]
    sql_query: Optional[str]
    sql_result: Optional[str]
    sql_retry_count: Optional[int]  # NEW — retry loop counter

print("✅ EnterpriseState schema defined (with sql_retry_count).")


✅ EnterpriseState schema defined (with sql_retry_count).


## 10. LLM Initialization

In [18]:
llm      = ChatGroq(model="llama-3.3-70b-versatile",api_key=groq_api_key_1)
llm_fast = ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api_key_1)   # routing / critic
print("✅ LLMs initialized: 70B (main) | 8B (router/critic)")


✅ LLMs initialized: 70B (main) | 8B (router/critic)


## 11. Text-to-SQL Engine V2 (Dual-DB + Retry Correction Loop)

Multi-database SQL agent that:
- Auto-detects which database to query based on table names
- Includes full join map in system prompt
- Retries up to 2 times with error feedback on SQL failures

In [19]:
CB_TABLES = {
    "cb_objects","cb_acquisitions","cb_ipos","cb_people",
    "cb_relationships","cb_offices","cb_milestones","cb_degrees",
    "cb_funds","cb_funding_rounds","cb_investments"
}

SQL_SYSTEM_PROMPT_V2 = f"""You are an expert enterprise analytics SQL agent with access to TWO SQLite databases.

═══ DATABASE 1: enterprise_intelligence.db (operational analytics) ═══
{DB_SCHEMA}

═══ DATABASE 2: crunchbase_ecosystem.db (startup ecosystem) ═══
{CB_SCHEMA}

═══ INSTRUCTIONS ═══
1. Think step-by-step. Identify which database the question touches.
2. For Crunchbase questions, use ONLY cb_* tables.
3. For multi-table Crunchbase queries, use the JOIN MAP above.
4. Write a single valid SQLite query. No markdown. No backticks. No explanation.
5. Always include LIMIT 25 unless the user asks for totals/aggregates.
6. For date arithmetic: strftime('%Y', date_column) to extract year.
7. For anomaly queries: compute AVG and STDEV in a subquery then filter.
8. NEVER hallucinate column names — only use columns from the schema above.
9. Use WITH (CTE) syntax for complex multi-step queries.

═══ EXAMPLE QUERIES ═══
Q: "Top 10 companies by total funding in enterprise software"
SQL:
SELECT name, funding_total_usd, status, country_code
FROM cb_objects
WHERE entity_type='Company' AND category_code='enterprise' AND funding_total_usd > 0
ORDER BY funding_total_usd DESC LIMIT 10;

Q: "Which acquirers did the most deals post-2010?"
SQL:
SELECT o.name, COUNT(*) as deal_count, SUM(a.price_amount) as total_spend_usd
FROM cb_acquisitions a
JOIN cb_objects o ON o.id = a.acquiring_object_id
WHERE a.acquired_at >= '2010-01-01'
GROUP BY o.id, o.name
ORDER BY deal_count DESC LIMIT 15;

Q: "Top sectors by total funding in enterprise DB"
SQL:
SELECT sector, SUM(amount_usd) as total_funding, COUNT(*) as deals
FROM startup_funding
GROUP BY sector ORDER BY total_funding DESC LIMIT 10;
"""


def execute_sql_safely_v2(sql: str) -> tuple:
    """Execute SQL against the appropriate DB; auto-detect from table names."""
    t0 = time.time()
    try:
        sql_clean = re.sub(r"```sql|```", "", sql).strip()
        uses_cb = any(t in sql_clean for t in CB_TABLES)
        db_path = CB_DB_PATH if uses_cb else DB_PATH
        conn = sqlite3.connect(db_path)
        df = pd.read_sql_query(sql_clean, conn)
        conn.close()
        latency = (time.time() - t0) * 1000
        if df.empty:
            return "Query returned no results.", latency
        return df.head(50).to_string(index=False), latency
    except Exception as e:
        return f"SQL ERROR: {e}", (time.time() - t0) * 1000


print("✅ SQL engine V2 ready (dual-DB + auto-detect).")


✅ SQL engine V2 ready (dual-DB + auto-detect).


## 12. Intent Router Node (V2 — 7 Routes)

In [20]:
ROUTER_PROMPT_V2 = """You are an enterprise analytics intent classifier.
Classify the user query into EXACTLY ONE of these routes:

- analytics  : SQL-answerable questions about structured data (funding, sales, KPIs, rankings, totals)
- rag        : Semantic search questions about knowledge base / company documents
- forecasting: Trend prediction, growth projection, time-series extrapolation
- anomaly    : Detecting outliers, spikes, failures, unusual patterns
- investor   : VC/investor analysis, fund sizes, investment patterns, which VCs to watch
- ecosystem  : Startup sector trends, geographic distribution, market analysis, exit rates
- general    : Conversation, greetings, out-of-scope questions

Reply with ONLY the route word. No explanation. No punctuation.
"""

VALID_ROUTES = {"analytics","rag","forecasting","anomaly","investor","ecosystem","general"}

def router_node(state: EnterpriseState) -> EnterpriseState:
    """Classify query into one of 7 routes."""
    t0 = time.time()
    query = state["messages"][-1].content
    resp = llm_fast.invoke([
        SystemMessage(content=ROUTER_PROMPT_V2),
        HumanMessage(content=query)
    ])
    route = resp.content.strip().lower()
    if route not in VALID_ROUTES:
        route = "general"
    print(f"🔀 Route: [{route.upper()}] | query: '{query[:60]}...'")
    return {
        **state,
        "route": route,
        "request_id": str(uuid.uuid4())[:8],
        "t_start": t0,
        "retrieval_ms": 0.0,
        "sql_ms": 0.0,
        "validator_passed": None,
        "confidence_score": None,
        "sources": [],
        "sql_query": None,
        "sql_result": None,
        "sql_retry_count": 0,
    }


def route_selector_v2(state: EnterpriseState):
    mapping = {
        "analytics":  "analytics_node",
        "rag":        "rag_node",
        "forecasting":"forecasting_node",
        "anomaly":    "anomaly_node",
        "investor":   "investor_agent_node",
        "ecosystem":  "ecosystem_agent_node",
        "general":    "general_node",
    }
    return mapping.get(state["route"], "general_node")


print("✅ Intent router V2 ready (7 routes).")


✅ Intent router V2 ready (7 routes).


## 13. Analytics / Text-to-SQL Agent (V2 — Retry Correction Loop)

SQL generation → execution → retry on error (max 2 retries with error feedback) → insight formatting.

In [21]:
def analytics_node(state: EnterpriseState) -> EnterpriseState:
    """Text-to-SQL with retry correction loop (up to 2 retries)."""
    query = state["messages"][-1].content
    MAX_RETRIES = 2
    generated_sql = None
    result_str = None
    sql_ms = 0.0
    retry_count = 0

    for attempt in range(MAX_RETRIES + 1):
        prompt = query if attempt == 0 else (
            f"{query}\n\nPrevious SQL attempt failed:\n{generated_sql}\n"
            f"Error: {result_str}\nPlease fix the SQL query."
        )
        sql_resp = llm.invoke([
            SystemMessage(content=SQL_SYSTEM_PROMPT_V2),
            HumanMessage(content=prompt)
        ])
        generated_sql = sql_resp.content.strip()
        result_str, sql_ms = execute_sql_safely_v2(generated_sql)
        if not result_str.startswith("SQL ERROR"):
            break
        retry_count += 1
        print(f"  ⚠️  SQL retry {retry_count}/{MAX_RETRIES}")

    formatter_prompt = f"""You are an enterprise BI analyst.
User asked: {query}
SQL: {generated_sql}
Results: {result_str[:1500]}

Provide: Key finding → 2-3 supporting metrics → 1 actionable recommendation.
"""
    insight = llm.invoke([HumanMessage(content=formatter_prompt)])

    db_used = "crunchbase" if any(t in generated_sql for t in CB_TABLES) else "enterprise"
    final = f"""📊 **Analytics Insight**

{insight.content}

---
*SQL:* `{generated_sql[:150]}{'...' if len(generated_sql)>150 else ''}`
*Execution:* {sql_ms:.1f}ms | *Retries:* {retry_count} | *DB:* {db_used}
"""
    return {
        **state,
        "messages": [*state["messages"], insight.__class__(content=final)],
        "sql_query": generated_sql,
        "sql_result": result_str,
        "sql_ms": sql_ms,
        "sql_retry_count": retry_count,
        "confidence_score": max(0.5, 0.85 - retry_count * 0.05),
        "sources": [f"{db_used}_db"],
    }

print("✅ Analytics node V2 ready (retry loop + dual-DB).")


✅ Analytics node V2 ready (retry loop + dual-DB).


## 14. Hybrid RAG Agent

In [22]:
def rag_node(state: EnterpriseState) -> EnterpriseState:
    """Hybrid retrieval (FAISS + BM25 + RRF) → grounded answer with citations."""
    query = state["messages"][-1].content
    t_ret = time.time()
    docs = hybrid_retrieve(query, top_k=4)
    retrieval_ms = (time.time() - t_ret) * 1000
    context = "\n\n".join([f"[Doc {i+1}]: {d.page_content}" for i, d in enumerate(docs)])
    sources = list(set(d.metadata.get("source", "kb") for d in docs))
    rag_prompt = f"""You are an enterprise knowledge analyst. Use ONLY the provided context to answer.

Context (hybrid FAISS + BM25 + RRF retrieval):
{context}

Question: {query}

Instructions:
- Ground every claim in the context above.
- If the context does not answer the question, say so clearly.
- Structure: Key Finding → Supporting Details → Implications.
- End with: Sources: {', '.join(sources)}
"""
    response = llm.invoke([HumanMessage(content=rag_prompt)])
    return {
        **state,
        "messages": [*state["messages"],
                     response.__class__(content=f"🔍 **Knowledge Retrieval**\n\n{response.content}")],
        "retrieval_ms": retrieval_ms,
        "confidence_score": 0.88,
        "sources": sources,
    }

print("✅ Hybrid RAG agent ready.")


✅ Hybrid RAG agent ready.


## 15. Forecasting Agent (KPI + Funding Trend)

Extends the original product KPI forecasting with sector-level funding trend
analysis using linear regression slope-per-year.

In [23]:
def forecast_funding_trends(sector_filter: str = None) -> str:
    """Sector-level funding trajectory analysis from Crunchbase data."""
    conn = sqlite3.connect(CB_DB_PATH)
    q = """
        SELECT strftime('%Y', first_funding_at) as year, category_code,
               COUNT(*) as companies_funded,
               SUM(funding_total_usd) as total_funding_usd,
               AVG(funding_total_usd) as avg_funding_usd
        FROM cb_objects
        WHERE entity_type='Company' AND first_funding_at IS NOT NULL
          AND first_funding_at != '' AND funding_total_usd > 0
    """
    if sector_filter:
        q += f" AND category_code = '{sector_filter}'"
    q += " GROUP BY year, category_code ORDER BY year"
    df = pd.read_sql_query(q, conn)
    conn.close()

    summaries = []
    for sector, group in df.groupby("category_code"):
        group = group.dropna(subset=["year"]).copy()
        group = group[group["year"].str.len() == 4]
        if len(group) < 3:
            continue
        group["t"] = range(len(group))
        coeffs = np.polyfit(group["t"], group["total_funding_usd"], 1)
        slope = coeffs[0]
        direction = "📈" if slope > 0 else "📉"
        last = group.iloc[-1]
        summaries.append(
            f"{direction} **{sector}**: ${last['total_funding_usd']:,.0f} in {last['year']}, "
            f"slope ${slope:+,.0f}/period, {last['companies_funded']:.0f} companies funded"
        )
    return "\n".join(summaries[:15]) if summaries else "Insufficient time-series data."


def forecasting_node(state: EnterpriseState) -> EnterpriseState:
    """Product KPI forecasting + Crunchbase funding trend forecasting."""
    query = state["messages"][-1].content
    t_sql = time.time()

    # ── KPI trend from enterprise DB ─────────────────────────────────────────
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT date, product, SUM(revenue) as total_revenue, AVG(dau) as avg_dau,
               AVG(churn_rate) as avg_churn
        FROM product_kpis GROUP BY date, product ORDER BY date
    """, conn)
    conn.close()
    sql_ms = (time.time() - t_sql) * 1000

    kpi_summary = []
    for product in df["product"].unique():
        pdata = df[df["product"] == product].copy()
        pdata["t"] = range(len(pdata))
        if len(pdata) < 4:
            continue
        coeffs = np.polyfit(pdata["t"], pdata["total_revenue"], 1)
        slope = coeffs[0]
        next_4_weeks = [coeffs[0]*(len(pdata)+i)+coeffs[1] for i in range(1, 5)]
        kpi_summary.append(
            f"**{product}**: {'📈' if slope>0 else '📉'} (slope: ${slope:+.0f}/wk). "
            f"4-week forecast: ${next_4_weeks[-1]:,.0f}"
        )

    # ── Funding trend from Crunchbase ─────────────────────────────────────────
    funding_trends = forecast_funding_trends()

    forecast_prompt = f"""You are a business forecasting analyst.

Product KPI Trends:
{'\n'.join(kpi_summary)}

Crunchbase Startup Funding Trends (by sector):
{funding_trends}

User Question: {query}

Provide an executive forecast summary:
1. Overall product revenue trajectory
2. Best/worst KPI momentum product
3. Top 3 sectors showing accelerating funding
4. Risk factors and recommended leadership actions
"""
    response = llm.invoke([HumanMessage(content=forecast_prompt)])
    final = f"📈 **Forecasting Analysis**\n\n{response.content}\n\n---\n*Sources: product_kpis + crunchbase_ecosystem | SQL: {sql_ms:.1f}ms*"
    return {
        **state,
        "messages": [*state["messages"], response.__class__(content=final)],
        "sql_ms": sql_ms,
        "confidence_score": 0.74,
        "sources": ["product_kpis","crunchbase_ecosystem"],
    }

print("✅ Forecasting agent ready (KPI + CB funding trends).")


✅ Forecasting agent ready (KPI + CB funding trends).


## 16. Anomaly Detection Agent (Ops + Funding + Acquisition Clustering)

Extended with:
- **Funding anomaly detection**: Z-score per sector peer group (not global)
- **Acquisition clustering**: Identifies acquirers with abnormal deal velocity

In [24]:
def detect_funding_anomalies() -> list:
    """
    Z-score funding outliers within each sector peer group.
    Peer-group normalization ensures a $500M raise in biotech is judged against biotech peers,
    not the whole market — dramatically reducing false positives.
    """
    conn = sqlite3.connect(CB_DB_PATH)
    df = pd.read_sql_query("""
        SELECT id, name, category_code, funding_total_usd, funding_rounds, founded_at, status
        FROM cb_objects
        WHERE entity_type='Company' AND funding_total_usd > 0 AND category_code != 'unknown'
    """, conn)
    conn.close()

    anomalies = []
    for sector, group in df.groupby("category_code"):
        if len(group) < 10:
            continue
        vals = group["funding_total_usd"].values
        mean, std = vals.mean(), vals.std()
        if std < 1:
            continue
        z = (vals - mean) / std
        for i, (idx, row) in enumerate(group.iterrows()):
            if abs(z[i]) > 3.0:
                anomalies.append({
                    "company": row["name"],
                    "sector": sector,
                    "funding_usd": row["funding_total_usd"],
                    "sector_mean_usd": round(mean, 0),
                    "z_score": round(float(z[i]), 2),
                    "status": row["status"],
                    "rounds": int(row["funding_rounds"]),
                })
    return sorted(anomalies, key=lambda x: abs(x["z_score"]), reverse=True)[:20]


def detect_acquisition_clusters() -> list:
    """Flag acquirers with abnormally high deal velocity (Z > 2.0)."""
    conn = sqlite3.connect(CB_DB_PATH)
    df = pd.read_sql_query("""
        SELECT acquiring_object_id, COUNT(*) as deals,
               MIN(acquired_at) as first_deal, MAX(acquired_at) as last_deal,
               SUM(price_amount) as total_spend
        FROM cb_acquisitions
        WHERE acquired_at IS NOT NULL AND acquired_at != ''
        GROUP BY acquiring_object_id HAVING deals >= 2
        ORDER BY deals DESC LIMIT 50
    """, conn)
    names = pd.read_sql_query(
        "SELECT id, name FROM cb_objects WHERE entity_type IN ('Company','FinancialOrg')", conn
    )
    conn.close()

    if df.empty:
        return []
    df = df.merge(names, left_on="acquiring_object_id", right_on="id", how="left")
    mean_d, std_d = df["deals"].mean(), df["deals"].std()
    if std_d < 0.01:
        return []
    df["z_score"] = ((df["deals"] - mean_d) / std_d).round(2)
    return df[df["z_score"] > 2.0].to_dict("records")


def anomaly_node(state: EnterpriseState) -> EnterpriseState:
    """Detect anomalies in operational metrics AND Crunchbase funding/acquisition data."""
    query = state["messages"][-1].content

    # ── Operational anomalies (original) ──────────────────────────────────────
    t_sql = time.time()
    conn = sqlite3.connect(DB_PATH)
    df_ops = pd.read_sql_query("SELECT * FROM operational_metrics", conn)
    conn.close()
    sql_ms = (time.time() - t_sql) * 1000

    ops_anomalies = []
    for svc in df_ops["service"].unique():
        svc_df = df_ops[df_ops["service"] == svc].copy()
        for col in ["cpu_pct","error_count","avg_latency_ms"]:
            vals = svc_df[col].values
            mean, std = vals.mean(), vals.std()
            if std < 1e-6:
                continue
            z_scores = np.abs((vals - mean) / std)
            anomaly_mask = z_scores > 2.5
            if anomaly_mask.sum() > 0:
                worst_idx = np.argmax(z_scores)
                ops_anomalies.append({
                    "service": svc, "metric": col,
                    "anomaly_count": int(anomaly_mask.sum()),
                    "worst_value": round(float(vals[worst_idx]), 2),
                    "mean": round(float(mean), 2),
                    "z_score": round(float(z_scores[worst_idx]), 2),
                    "timestamp": svc_df.iloc[worst_idx]["timestamp"]
                })

    # ── Funding anomalies (new) ────────────────────────────────────────────────
    funding_anomalies = detect_funding_anomalies()
    acq_clusters = detect_acquisition_clusters()

    anomaly_prompt = f"""You are an SRE and financial analyst detecting anomalies across two systems.

OPERATIONAL ANOMALIES (Z-score > 2.5σ):
{json.dumps(ops_anomalies[:5], indent=2) if ops_anomalies else "None detected."}

FUNDING ANOMALIES (sector peer Z-score > 3.0):
{json.dumps(funding_anomalies[:5], indent=2) if funding_anomalies else "None detected."}

ACQUISITION CLUSTERING (deal velocity Z-score > 2.0):
{json.dumps(acq_clusters[:3], indent=2) if acq_clusters else "None detected."}

User Question: {query}

Provide:
1. Severity assessment per domain (Critical / High / Medium)
2. Most concerning operational signal and root cause hypothesis
3. Most anomalous funding situation and what it could signal
4. Serial acquirer pattern and strategic interpretation
5. Immediate action items
"""
    response = llm.invoke([HumanMessage(content=anomaly_prompt)])
    total_anomalies = len(ops_anomalies) + len(funding_anomalies) + len(acq_clusters)
    final = (f"🚨 **Anomaly Detection Report**\n\n{response.content}\n\n---\n"
             f"*{len(ops_anomalies)} ops | {len(funding_anomalies)} funding | "
             f"{len(acq_clusters)} acquisition cluster anomalies | SQL: {sql_ms:.1f}ms*")
    return {
        **state,
        "messages": [*state["messages"], response.__class__(content=final)],
        "sql_ms": sql_ms,
        "confidence_score": 0.79,
        "sources": ["operational_metrics","crunchbase_ecosystem"],
    }

print("✅ Extended anomaly detection agent ready.")


✅ Extended anomaly detection agent ready.


## 17. Investor Intelligence Agent

In [25]:
def investor_agent_node(state: EnterpriseState) -> EnterpriseState:
    """
    Handles VC/investor questions by combining:
    - SQL: structured fund sizes and deal counts from cb_funds + cb_objects
    - RAG: investor narratives from the extended knowledge corpus
    """
    query = state["messages"][-1].content

    # Structured investor metrics
    investor_sql = """
        SELECT o.name, o.status, o.country_code,
               COUNT(f.fund_id) as num_funds,
               SUM(f.raised_amount) as total_raised,
               AVG(f.raised_amount) as avg_fund_size,
               o.description
        FROM cb_objects o
        LEFT JOIN cb_funds f ON o.id = f.object_id
        WHERE o.entity_type = 'FinancialOrg'
        GROUP BY o.id, o.name
        ORDER BY total_raised DESC
        LIMIT 20
    """
    sql_result, sql_ms = execute_sql_safely_v2(investor_sql)

    # RAG: investor narratives
    t_ret = time.time()
    docs = hybrid_retrieve(f"investor venture capital {query}", top_k=3)
    retrieval_ms = (time.time() - t_ret) * 1000
    rag_context = "\n".join(d.page_content for d in docs)

    combined_prompt = f"""You are an investor intelligence analyst.

Structured Data (top investors by capital raised):
{sql_result}

Knowledge Context (from corpus):
{rag_context}

User Question: {query}

Answer with:
1. Investor tier analysis (top-tier vs emerging)
2. Notable fund sizes and recent activity
3. Geographic investment concentration
4. Investment focus areas by sector
5. One actionable insight for an entrepreneur or LP
Ground every claim in the data above.
"""
    response = llm.invoke([HumanMessage(content=combined_prompt)])
    final = (f"💼 **Investor Intelligence**\n\n{response.content}\n\n---\n"
             f"*SQL: {sql_ms:.1f}ms | RAG: 3 docs ({retrieval_ms:.0f}ms)*")
    return {
        **state,
        "messages": [*state["messages"], response.__class__(content=final)],
        "sql_ms": sql_ms,
        "retrieval_ms": retrieval_ms,
        "confidence_score": 0.83,
        "sources": ["cb_funds","cb_objects","crunchbase_rag"],
    }

print("✅ Investor intelligence agent ready.")


✅ Investor intelligence agent ready.


## 18. Ecosystem Intelligence Agent

In [26]:
def ecosystem_agent_node(state: EnterpriseState) -> EnterpriseState:
    """
    Handles broad ecosystem questions: sector analysis, geographic trends,
    startup status distributions, and acquisition/IPO exit rate analysis.
    """
    query = state["messages"][-1].content

    eco_sql = """
        SELECT category_code,
               COUNT(*) as company_count,
               SUM(CASE WHEN status='acquired' THEN 1 ELSE 0 END) as acquisitions,
               SUM(CASE WHEN status='ipo' THEN 1 ELSE 0 END) as ipos,
               SUM(CASE WHEN status='closed' THEN 1 ELSE 0 END) as closures,
               AVG(funding_total_usd) as avg_funding,
               SUM(funding_total_usd) as total_funding
        FROM cb_objects
        WHERE entity_type='Company' AND category_code != 'unknown' AND category_code IS NOT NULL
        GROUP BY category_code
        ORDER BY company_count DESC LIMIT 20
    """
    sql_result, sql_ms = execute_sql_safely_v2(eco_sql)

    # Geographic breakdown
    geo_sql = """
        SELECT country_code, COUNT(*) as companies,
               SUM(funding_total_usd) as total_funding,
               AVG(funding_total_usd) as avg_funding
        FROM cb_objects
        WHERE entity_type='Company' AND country_code != 'UNKNOWN'
        GROUP BY country_code ORDER BY total_funding DESC LIMIT 15
    """
    geo_result, _ = execute_sql_safely_v2(geo_sql)

    eco_prompt = f"""You are a startup ecosystem analyst.

Sector Data (companies by category):
{sql_result}

Geographic Distribution (top countries by funding):
{geo_result}

User Question: {query}

Provide:
1. Sector comparison: which sectors lead in volume vs funding
2. Exit rate analysis: acquisition vs IPO vs closure by sector
3. Geographic concentration: dominant hubs and emerging markets
4. Emerging sector signals
5. One strategic observation for investors or founders
"""
    response = llm.invoke([HumanMessage(content=eco_prompt)])
    final = (f"🌐 **Ecosystem Intelligence**\n\n{response.content}\n\n---\n"
             f"*Source: cb_objects sector + geo rollup | SQL: {sql_ms:.1f}ms*")
    return {
        **state,
        "messages": [*state["messages"], response.__class__(content=final)],
        "sql_ms": sql_ms,
        "confidence_score": 0.81,
        "sources": ["cb_objects"],
    }

print("✅ Ecosystem intelligence agent ready.")


✅ Ecosystem intelligence agent ready.


## 19. General Conversation Node

In [27]:
def general_node(state: EnterpriseState) -> EnterpriseState:
    """Fallback for greetings and out-of-scope queries."""
    system = SystemMessage(content="""You are ARIA — an Enterprise Decision Intelligence assistant.
You help analysts, executives, and data teams query enterprise and startup ecosystem data,
detect anomalies, generate forecasts, and retrieve business knowledge.
You have access to:
- Operational enterprise data (startup funding, sales pipeline, product KPIs, ops metrics)
- Crunchbase startup ecosystem (companies, investors, acquisitions, IPOs, people, milestones)
""")
    messages = [system] + list(state["messages"])
    response = llm.invoke(messages)
    return {
        **state,
        "messages": [*state["messages"], response],
        "confidence_score": 0.95,
        "sources": [],
    }

print("✅ General conversation node ready.")


✅ General conversation node ready.


## 20. Critic / Validator Node

In [28]:
CRITIC_PROMPT = """You are a rigorous enterprise AI validator. Evaluate the assistant's response.

Check for:
1. HALLUCINATION: Does the response make claims not supported by data/context?
2. SQL ACCURACY: If SQL was used, is the interpretation of results correct?
3. GROUNDING: Are claims attributed to sources?
4. CONFIDENCE: Does confidence level match evidence strength?

Output ONLY valid JSON in this format:
{
  "passed": true or false,
  "confidence": 0.0 to 1.0,
  "issues": ["issue1"] or [],
  "verdict": "one sentence verdict"
}
"""

def critic_node(state: EnterpriseState) -> EnterpriseState:
    """Validate last assistant response. Fail-open: exceptions pass through."""
    last_response = state["messages"][-1].content if state["messages"] else ""
    sql_result = state.get("sql_result","N/A") or "N/A"
    route = state.get("route","general")
    critic_input = f"""Route: {route}
Last Response (first 600 chars): {last_response[:600]}
SQL Result (if any, first 300 chars): {str(sql_result)[:300]}
"""
    try:
        resp = llm_fast.invoke([
            SystemMessage(content=CRITIC_PROMPT),
            HumanMessage(content=critic_input)
        ])
        text = re.sub(r"```json|```", "", resp.content.strip()).strip()
        verdict = json.loads(text)
        passed = verdict.get("passed", True)
        confidence = float(verdict.get("confidence", state.get("confidence_score", 0.8) or 0.8))
        issues = verdict.get("issues", [])
        if issues:
            print(f"  ⚠️  Critic flagged: {issues}")
        else:
            print(f"  ✅ Critic passed (confidence: {confidence:.2f})")
    except Exception:
        passed = True
        confidence = state.get("confidence_score", 0.8) or 0.8
    return {**state, "validator_passed": passed, "confidence_score": confidence}

print("✅ Critic/Validator node ready.")


✅ Critic/Validator node ready.


## 21. Telemetry Finalizer Node

In [29]:
def telemetry_node(state: EnterpriseState) -> EnterpriseState:
    """Log complete telemetry record with retry count, agent type, and DB queried."""
    t_end = time.time()
    t_start = state.get("t_start") or t_end
    total_ms = (t_end - t_start) * 1000
    query = ""
    for msg in reversed(state["messages"]):
        if hasattr(msg, "type") and msg.type == "human":
            query = msg.content
            break
    last_response = state["messages"][-1].content if state["messages"] else ""
    est_tokens = max(1, len(last_response.split()) * 4 // 3)
    route = state.get("route","general")
    db_queried = "crunchbase" if route in ["ecosystem","investor"] else (
        "both" if route in ["forecasting","anomaly"] else "enterprise"
    )
    record = TelemetryRecord(
        request_id=state.get("request_id","unknown"),
        timestamp=datetime.now().isoformat(),
        query=query[:80],
        route=route,
        total_latency_ms=round(total_ms, 1),
        retrieval_latency_ms=round(state.get("retrieval_ms",0) or 0, 1),
        sql_latency_ms=round(state.get("sql_ms",0) or 0, 1),
        llm_latency_ms=round(total_ms-(state.get("retrieval_ms",0) or 0)-(state.get("sql_ms",0) or 0), 1),
        estimated_tokens=est_tokens,
        confidence_score=round(state.get("confidence_score",0.8) or 0.8, 3),
        sources_cited=len(state.get("sources",[]) or []),
        validator_passed=state.get("validator_passed", True),
        sql_retry_count=state.get("sql_retry_count",0) or 0,
        agent_type=route,
        db_queried=db_queried,
        error=None,
    )
    log_telemetry(record)
    print(f"  📊 Telemetry | {total_ms:.0f}ms | route:{record.route} | "
          f"conf:{record.confidence_score:.2f} | retries:{record.sql_retry_count} | db:{db_queried}")
    return state

print("✅ Extended telemetry finalizer ready.")


✅ Extended telemetry finalizer ready.


## 22. LangGraph Assembly (9-Node Graph)

Wire all nodes into the production state graph with conditional routing
from the 7-route intent classifier.

In [30]:
checkpointer = MemorySaver()
graph = StateGraph(EnterpriseState)

# ── Register all nodes ────────────────────────────────────────────────────────
graph.add_node("router_node",          router_node)
graph.add_node("analytics_node",       analytics_node)
graph.add_node("rag_node",             rag_node)
graph.add_node("forecasting_node",     forecasting_node)
graph.add_node("anomaly_node",         anomaly_node)
graph.add_node("general_node",         general_node)
graph.add_node("investor_agent_node",  investor_agent_node)
graph.add_node("ecosystem_agent_node", ecosystem_agent_node)
graph.add_node("critic_node",          critic_node)
graph.add_node("telemetry_node",       telemetry_node)

# ── Entry ──────────────────────────────────────────────────────────────────────
graph.add_edge(START, "router_node")

# ── Conditional dispatch ───────────────────────────────────────────────────────
graph.add_conditional_edges(
    "router_node",
    route_selector_v2,
    {
        "analytics_node":      "analytics_node",
        "rag_node":            "rag_node",
        "forecasting_node":    "forecasting_node",
        "anomaly_node":        "anomaly_node",
        "investor_agent_node": "investor_agent_node",
        "ecosystem_agent_node":"ecosystem_agent_node",
        "general_node":        "general_node",
    }
)

# ── All agents → critic → telemetry → END ─────────────────────────────────────
ALL_AGENTS = [
    "analytics_node","rag_node","forecasting_node","anomaly_node",
    "general_node","investor_agent_node","ecosystem_agent_node"
]
for agent in ALL_AGENTS:
    graph.add_edge(agent, "critic_node")

graph.add_edge("critic_node",   "telemetry_node")
graph.add_edge("telemetry_node", END)

platform = graph.compile(checkpointer=checkpointer)
print("✅ ARIA Enterprise Intelligence Graph compiled.")
print(f"   Nodes: router | analytics | rag | forecasting | anomaly | investor | ecosystem | critic | telemetry")
print(f"   Routes: analytics | rag | forecasting | anomaly | investor | ecosystem | general")


✅ ARIA Enterprise Intelligence Graph compiled.
   Nodes: router | analytics | rag | forecasting | anomaly | investor | ecosystem | critic | telemetry
   Routes: analytics | rag | forecasting | anomaly | investor | ecosystem | general


## 23. CLI Query Interface & Smoke Tests

In [31]:
def query_platform(user_input: str, thread_id: str = "default") -> str:
    """Run a single query through the enterprise platform."""
    config = {"configurable": {"thread_id": thread_id}}
    result = platform.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    for msg in reversed(result["messages"]):
        if not isinstance(msg, HumanMessage):
            return msg.content
    return "No response generated."


print("\n" + "="*65)
print("🔥 ARIA ENTERPRISE INTELLIGENCE — SMOKE TESTS")
print("="*65)

smoke_tests = [
    ("Which sector raised the most total funding?",          "analytics"),
    ("What are the best practices for RAG architecture?",    "rag"),
    ("Forecast revenue trends for our products",             "forecasting"),
    ("Are there anomalies in our operational metrics?",      "anomaly"),
    ("Which VC funds raised the most capital?",              "investor"),
    ("What sectors have the highest acquisition exit rate?", "ecosystem"),
]
for q, expected in smoke_tests:
    print(f"\n❓ [{expected.upper()}] {q}")
    print("-" * 55)
    resp = query_platform(q, thread_id="smoke_test")
    print(resp[:250], "...")



🔥 ARIA ENTERPRISE INTELLIGENCE — SMOKE TESTS

❓ [ANALYTICS] Which sector raised the most total funding?
-------------------------------------------------------
🔀 Route: [ANALYTICS] | query: 'Which sector raised the most total funding?...'
  ✅ Critic passed (confidence: 1.00)
  📊 Telemetry | 2514ms | route:analytics | conf:1.00 | retries:0 | db:enterprise
📊 **Analytics Insight**

**Key Finding:** The Cloud sector raised the most total funding, with a staggering $400 million in just 2 deals.

**Supporting Metrics:**
1. **Funding per Deal:** The Cloud sector has an average funding per deal of $200 milli ...

❓ [RAG] What are the best practices for RAG architecture?
-------------------------------------------------------
🔀 Route: [RAG] | query: 'What are the best practices for RAG architecture?...'
  ✅ Critic passed (confidence: 0.90)
  📊 Telemetry | 1034ms | route:rag | conf:0.90 | retries:0 | db:enterprise
🔍 **Knowledge Retrieval**

Key Finding: The best practices for RAG (Retrieval-Aug

## 24. Evaluation Framework

Benchmark suite covering routing accuracy, SQL success rate, retry frequency,
validator pass rate, latency metrics, and confidence scoring.

In [32]:
QUERY_TEST_SUITE = [
    # Text-to-SQL — enterprise DB
    {"id":"sql-01","query":"Which sector raised the most total funding?",
     "expected_route":"analytics","check":"contains_number"},
    {"id":"sql-02","query":"What is the average deal value by sales region?",
     "expected_route":"analytics","check":"contains_number"},
    {"id":"sql-03","query":"Which product has the highest average DAU?",
     "expected_route":"analytics","check":"contains_number"},
    # Text-to-SQL — Crunchbase
    {"id":"sql-04","query":"Top 10 companies by total funding in the AI sector",
     "expected_route":"analytics","check":"contains_number"},
    {"id":"sql-05","query":"Which acquirers completed the most deals post-2010?",
     "expected_route":"analytics","check":"contains_number"},
    {"id":"sql-06","query":"IPO companies with highest valuations",
     "expected_route":"analytics","check":"contains_number"},
    # Routing accuracy
    {"id":"route-01","query":"Tell me about Y Combinator and its portfolio",
     "expected_route":"rag"},
    {"id":"route-02","query":"Which VC raised the most capital?",
     "expected_route":"investor"},
    {"id":"route-03","query":"Detect unusual funding patterns in biotech",
     "expected_route":"anomaly"},
    {"id":"route-04","query":"What sectors have the highest IPO exit rates?",
     "expected_route":"ecosystem"},
    {"id":"route-05","query":"Forecast startup funding trends for next year",
     "expected_route":"forecasting"},
    # RAG grounding
    {"id":"rag-01","query":"What are best practices for LangGraph multi-agent systems?",
     "expected_route":"rag"},
    {"id":"rag-02","query":"What does enterprise AI adoption look like in 2024?",
     "expected_route":"rag"},
    # Investor
    {"id":"inv-01","query":"Compare fund sizes across top-tier VCs",
     "expected_route":"investor"},
    # Ecosystem
    {"id":"eco-01","query":"What is the geographic distribution of funded startups?",
     "expected_route":"ecosystem"},
]


def run_evaluation(test_suite: list = None) -> tuple:
    """Run the benchmark test suite and compute aggregate metrics."""
    if test_suite is None:
        test_suite = QUERY_TEST_SUITE
    results = []
    print(f"🧪 Running {len(test_suite)} evaluation queries...")
    for test in test_suite:
        start = time.time()
        try:
            output = platform.invoke(
                {"messages": [HumanMessage(content=test["query"])]},
                config={"configurable": {"thread_id": test["id"]}}
            )
            latency_ms = (time.time() - start) * 1000
            actual_route = output.get("route","unknown")
            sql_error = str(output.get("sql_result","") or "").startswith("SQL ERROR")
            results.append({
                "test_id": test["id"],
                "query": test["query"][:60],
                "expected_route": test.get("expected_route","any"),
                "actual_route": actual_route,
                "route_correct": (test.get("expected_route","") == actual_route),
                "sql_success": not sql_error,
                "sql_retries": output.get("sql_retry_count", 0) or 0,
                "validator_passed": output.get("validator_passed", True),
                "confidence_score": round(output.get("confidence_score", 0.8) or 0.8, 3),
                "latency_ms": round(latency_ms, 1),
                "sources_cited": len(output.get("sources",[]) or []),
                "error": None,
            })
            print(f"  [{actual_route.upper():<12}] {test['id']:<12} {latency_ms:.0f}ms "
                  f"{'✅' if results[-1]['route_correct'] else '❌'}")
        except Exception as e:
            results.append({"test_id": test["id"], "query": test["query"][:60],
                            "error": str(e), "latency_ms": 0, "route_correct": False,
                            "sql_success": False, "sql_retries": 0, "validator_passed": False,
                            "confidence_score": 0.0, "sources_cited": 0,
                            "expected_route": test.get("expected_route",""), "actual_route": "error"})
            print(f"  [ERROR     ] {test['id']} — {e}")

    df = pd.DataFrame(results)
    metrics = {}
    for col, label in [
        ("route_correct",    "routing_accuracy"),
        ("sql_success",      "sql_execution_success_rate"),
        ("validator_passed", "validator_pass_rate"),
        ("confidence_score", "avg_confidence"),
    ]:
        if col in df.columns:
            metrics[label] = round(df[col].mean(), 3)
    if "sql_retries" in df.columns:
        metrics["avg_sql_retry_count"] = round(df["sql_retries"].mean(), 3)
    if "latency_ms" in df.columns:
        metrics["p50_latency_ms"] = round(df["latency_ms"].quantile(0.50), 1)
        metrics["p95_latency_ms"] = round(df["latency_ms"].quantile(0.95), 1)
    if "sources_cited" in df.columns:
        metrics["avg_sources_per_response"] = round(df["sources_cited"].mean(), 2)

    print("\n📊 ARIA EVALUATION METRICS:")
    print("-" * 45)
    for k, v in metrics.items():
        print(f"  {k:<35}: {v}")
    print("-" * 45)
    return df, metrics


eval_df, eval_metrics = run_evaluation(QUERY_TEST_SUITE[:6])  # Run subset for speed
print("\n✅ Evaluation complete.")
print(eval_df[["test_id","expected_route","actual_route","route_correct","latency_ms"]].to_string(index=False))


🧪 Running 6 evaluation queries...
🔀 Route: [ANALYTICS] | query: 'Which sector raised the most total funding?...'
  ⚠️  Critic flagged: ["HALLUCINATION: The response makes claims not supported by data/context (e.g., 'staggering', 'strong investor interest', 'high growth potential', and 'market validation' are unsubstantiated assertions)"]
  📊 Telemetry | 2128ms | route:analytics | conf:0.60 | retries:0 | db:enterprise
  [ANALYTICS   ] sql-01       2132ms ✅
🔀 Route: [ANALYTICS] | query: 'What is the average deal value by sales region?...'
  ✅ Critic passed (confidence: 0.95)
  📊 Telemetry | 2297ms | route:analytics | conf:0.95 | retries:0 | db:enterprise
  [ANALYTICS   ] sql-02       2300ms ✅
🔀 Route: [ANALYTICS] | query: 'Which product has the highest average DAU?...'
  ✅ Critic passed (confidence: 0.90)
  📊 Telemetry | 2154ms | route:analytics | conf:0.90 | retries:0 | db:enterprise
  [ANALYTICS   ] sql-03       2156ms ✅
🔀 Route: [ANALYTICS] | query: 'Top 10 companies by total funding 

## 25. Gradio Intelligence Dashboard

Extended dashboard with 6 tabs:
1. Intelligence Chat — full multi-agent chat interface
2. Analytics Dashboard — enterprise KPI charts
3. Startup Intelligence — Crunchbase sector/country explorer
4. Acquisition Intelligence — M&A trends + anomaly table
5. Observability & Telemetry — latency, routes, retry counts
6. System Architecture — component reference

In [33]:
# ── Chart builders ────────────────────────────────────────────────────────────

def build_kpi_chart():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT date, product, SUM(revenue) as revenue
        FROM product_kpis GROUP BY date, product ORDER BY date
    """, conn)
    conn.close()
    fig = px.line(df, x="date", y="revenue", color="product",
                  title="Product Revenue Trend (Weekly)", template="plotly_dark",
                  color_discrete_sequence=["#00d4ff","#7c3aed","#10b981"])
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return fig

def build_pipeline_chart():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT stage, COUNT(*) as deals, SUM(deal_value) as total_value
        FROM sales_pipeline GROUP BY stage
    """, conn)
    conn.close()
    fig = px.bar(df, x="stage", y="total_value", color="deals",
                 title="Sales Pipeline Value by Stage", template="plotly_dark",
                 color_continuous_scale="Teal")
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return fig

def build_funding_chart():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("""
        SELECT sector, SUM(amount_usd)/1e6 as total_m
        FROM startup_funding GROUP BY sector ORDER BY total_m DESC
    """, conn)
    conn.close()
    fig = px.bar(df, x="total_m", y="sector", orientation="h",
                 title="Enterprise Funding by Sector ($M)", template="plotly_dark",
                 color="total_m", color_continuous_scale="Viridis")
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return fig

def build_telemetry_chart():
    df = get_telemetry_df()
    if df.empty:
        fig = go.Figure()
        fig.update_layout(title="No telemetry yet — run some queries!", template="plotly_dark")
        return fig
    fig = px.scatter(df, x="timestamp", y="total_latency_ms", color="route",
                     size="estimated_tokens", title="Request Latency by Route",
                     template="plotly_dark",
                     hover_data=["confidence_score","validator_passed","query","sql_retry_count"])
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return fig

def build_route_pie():
    df = get_telemetry_df()
    if df.empty:
        fig = go.Figure()
        fig.update_layout(title="No data yet", template="plotly_dark")
        return fig
    route_counts = df["route"].value_counts().reset_index()
    route_counts.columns = ["route","count"]
    fig = px.pie(route_counts, values="count", names="route",
                 title="Route Distribution", template="plotly_dark",
                 color_discrete_sequence=px.colors.sequential.Teal)
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)")
    return fig

def build_retry_chart():
    """Show SQL retry counts per route."""
    df = get_telemetry_df()
    if df.empty or "sql_retry_count" not in df.columns:
        return go.Figure().update_layout(title="No retry data yet", template="plotly_dark")
    retry_df = df.groupby("route")["sql_retry_count"].mean().reset_index()
    fig = px.bar(retry_df, x="route", y="sql_retry_count",
                 title="Avg SQL Retries by Route", template="plotly_dark",
                 color="sql_retry_count", color_continuous_scale="Reds")
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return fig


# ── Startup Intelligence helpers ───────────────────────────────────────────────

def get_all_sectors() -> list:
    try:
        conn = sqlite3.connect(CB_DB_PATH)
        df = pd.read_sql_query("""
            SELECT DISTINCT category_code FROM cb_objects
            WHERE entity_type='Company' AND category_code != 'unknown'
              AND category_code IS NOT NULL
            ORDER BY category_code LIMIT 50
        """, conn)
        conn.close()
        return df["category_code"].tolist()
    except Exception:
        return ["web","mobile","enterprise","biotech","cleantech","fintech","ai","saas"]


def run_ecosystem_query(sector, status, country):
    try:
        conn = sqlite3.connect(CB_DB_PATH)
        where = ["entity_type='Company'"]
        if sector and sector != "All":
            where.append(f"category_code='{sector}'")
        if status and status != "All":
            where.append(f"status='{status}'")
        if country and country.strip():
            where.append(f"country_code='{country.strip().upper()}'")
        where_str = " AND ".join(where)
        df = pd.read_sql_query(f"""
            SELECT category_code, status, country_code,
                   COUNT(*) as company_count,
                   ROUND(AVG(funding_total_usd)/1e6, 2) as avg_funding_m,
                   ROUND(SUM(funding_total_usd)/1e6, 2) as total_funding_m
            FROM cb_objects WHERE {where_str}
            GROUP BY category_code, status, country_code
            ORDER BY total_funding_m DESC LIMIT 50
        """, conn)
        conn.close()
        fig = px.bar(df.head(20), x="category_code", y="total_funding_m", color="status",
                     title="Total Funding by Sector & Status ($M)", template="plotly_dark",
                     labels={"total_funding_m":"Total Funding ($M)","category_code":"Sector"},
                     color_discrete_sequence=px.colors.qualitative.Vivid)
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
        return df, fig
    except Exception as e:
        return pd.DataFrame({"error":[str(e)]}), go.Figure()


def load_acquisition_data():
    try:
        conn = sqlite3.connect(CB_DB_PATH)
        df = pd.read_sql_query("""
            SELECT o.name, COUNT(*) as deals,
                   ROUND(SUM(a.price_amount)/1e6, 2) as total_spend_m,
                   ROUND(AVG(a.price_amount)/1e6, 2) as avg_deal_m,
                   MIN(a.acquired_at) as first_deal,
                   MAX(a.acquired_at) as latest_deal
            FROM cb_acquisitions a
            JOIN cb_objects o ON o.id = a.acquiring_object_id
            GROUP BY o.id, o.name ORDER BY deals DESC LIMIT 30
        """, conn)
        conn.close()
        clusters = pd.DataFrame(detect_acquisition_clusters())
        return df, clusters if not clusters.empty else pd.DataFrame({"message":["No anomalous acquirers detected."]})
    except Exception as e:
        return pd.DataFrame({"error":[str(e)]}), pd.DataFrame()


def load_funding_anomalies():
    try:
        anomalies = detect_funding_anomalies()
        return pd.DataFrame(anomalies) if anomalies else pd.DataFrame({"message":["No funding anomalies detected."]})
    except Exception as e:
        return pd.DataFrame({"error":[str(e)]})


# ── Chat logic ─────────────────────────────────────────────────────────────────
chat_thread_id = "gradio_session"

def chat_fn(message, history):
    return query_platform(message, thread_id=chat_thread_id)

def refresh_telemetry():
    df = get_telemetry_df()
    if df.empty:
        return "No telemetry yet.", build_telemetry_chart(), build_route_pie(), build_retry_chart()
    cols = ["timestamp","route","total_latency_ms","confidence_score",
            "validator_passed","sql_retry_count","db_queried","query"]
    available = [c for c in cols if c in df.columns]
    summary = df[available].tail(10).to_string(index=False)
    return summary, build_telemetry_chart(), build_route_pie(), build_retry_chart()


# ── CSS ────────────────────────────────────────────────────────────────────────
css = """
body { background: #0a0a0f; }
.gradio-container { font-family: 'JetBrains Mono', monospace; background: #0d0d1a; }
h1, h2, h3 { color: #00d4ff; }
"""

# ── Dashboard Assembly ─────────────────────────────────────────────────────────
with gr.Blocks(
    title="ARIA — Enterprise Intelligence Platform",
    theme=gr.themes.Base(primary_hue="cyan", neutral_hue="slate"),
    css=css
) as dashboard:

    gr.Markdown("""
    # 🏢 ARIA — Enterprise Intelligence & Agentic Analytics Platform
    **Powered by:** LangGraph · Hybrid RAG (FAISS + BM25 + RRF) · Text-to-SQL (Dual-DB) ·
    Crunchbase Ecosystem · Anomaly Detection · Forecasting · Critic/Validator · Telemetry
    """)

    with gr.Tabs():

        # ── Tab 1: Intelligence Chat ──────────────────────────────────────────
        with gr.Tab("🤖 Intelligence Chat"):
            gr.Markdown("### Ask anything about enterprise or startup ecosystem data")
            gr.Markdown(
                "**Try:**\n"
                "- *'Which sector raised the most funding?'*\n"
                "- *'Which VC funds raised the most capital?'*\n"
                "- *'What sectors have the highest acquisition exit rates?'*\n"
                "- *'Are there any anomalies in our operational metrics?'*\n"
                "- *'Forecast startup funding trends'*"
            )
            gr.ChatInterface(
                fn=chat_fn,
                type="messages",
                examples=[
                    "Which startup raised the most funding?",
                    "Which VC raised the most capital?",
                    "What sectors have the highest IPO exit rates?",
                    "Detect anomalies in funding and operational data",
                    "Forecast product revenue and startup funding trends",
                    "What are best practices for RAG architecture?",
                ]
            )

        # ── Tab 2: Analytics Dashboard ────────────────────────────────────────
        with gr.Tab("📊 Analytics Dashboard"):
            gr.Markdown("### Enterprise KPI Visualizations")
            with gr.Row():
                kpi_plot      = gr.Plot(value=build_kpi_chart,      label="Revenue Trend")
                funding_plot  = gr.Plot(value=build_funding_chart,  label="Funding by Sector")
            with gr.Row():
                pipeline_plot = gr.Plot(value=build_pipeline_chart, label="Sales Pipeline")
            refresh_btn = gr.Button("🔄 Refresh Charts")
            refresh_btn.click(
                fn=lambda: (build_kpi_chart(), build_funding_chart(), build_pipeline_chart()),
                outputs=[kpi_plot, funding_plot, pipeline_plot]
            )

        # ── Tab 3: Startup Intelligence ───────────────────────────────────────
        with gr.Tab("🚀 Startup Intelligence"):
            gr.Markdown("## Crunchbase Ecosystem Explorer")
            with gr.Row():
                sector_dd  = gr.Dropdown(choices=["All"]+get_all_sectors(),
                                         label="Filter by Sector", value="All")
                status_dd  = gr.Dropdown(choices=["All","operating","acquired","ipo","closed"],
                                         label="Company Status", value="All")
                country_tb = gr.Textbox(label="Country Code (e.g. USA)", value="")
            search_btn    = gr.Button("🔍 Run Ecosystem Query")
            stats_output  = gr.DataFrame(label="Sector × Status × Country Statistics")
            chart_output  = gr.Plot(label="Funding Distribution")
            search_btn.click(
                fn=run_ecosystem_query,
                inputs=[sector_dd, status_dd, country_tb],
                outputs=[stats_output, chart_output]
            )

        # ── Tab 4: Acquisition Intelligence ──────────────────────────────────
        with gr.Tab("🤝 Acquisition Intelligence"):
            gr.Markdown("## M&A Trend Analysis")
            run_acq_btn   = gr.Button("📊 Load Acquisition Data")
            acq_table     = gr.DataFrame(label="Top Acquirers by Deal Count")
            anomaly_table = gr.DataFrame(label="Anomalous Acquirers (Deal Velocity Z > 2.0)")
            funding_anom  = gr.DataFrame(label="Funding Anomalies (Sector Peer Z > 3.0)")
            run_acq_btn.click(
                fn=lambda: (*load_acquisition_data(), load_funding_anomalies()),
                outputs=[acq_table, anomaly_table, funding_anom]
            )

        # ── Tab 5: Observability & Telemetry ──────────────────────────────────
        with gr.Tab("🔬 Observability & Telemetry"):
            gr.Markdown("### Production Telemetry — Latency · Confidence · Route · Retries · DB")
            with gr.Row():
                latency_plot  = gr.Plot(label="Latency by Route")
                route_plot    = gr.Plot(label="Route Distribution")
            retry_plot    = gr.Plot(label="Avg SQL Retries by Route")
            telemetry_table = gr.Textbox(label="Recent Requests (last 10)", lines=12, interactive=False)
            refresh_telem_btn = gr.Button("🔄 Refresh Telemetry")
            refresh_telem_btn.click(
                fn=refresh_telemetry,
                outputs=[telemetry_table, latency_plot, route_plot, retry_plot]
            )

        # ── Tab 6: System Architecture ────────────────────────────────────────
        with gr.Tab("⚙️ System Architecture"):
            gr.Markdown("""
## ARIA Extended Architecture

```
[START]
   │
[Intent Router] ──── LLaMA 8B (7-route classification)
   │
   ├── analytics    ──► [Text-to-SQL V2]        ──► enterprise_intelligence.db + crunchbase_ecosystem.db
   ├── rag          ──► [Hybrid RAG Agent]       ──► FAISS + BM25 + RRF (enterprise + startup corpus)
   ├── forecasting  ──► [Forecasting Agent]      ──► product_kpis + CB funding trends
   ├── anomaly      ──► [Anomaly Agent]          ──► ops Z-score + sector peer funding + acq clustering
   ├── investor     ──► [Investor Agent]         ──► cb_funds + cb_objects + RAG
   ├── ecosystem    ──► [Ecosystem Agent]        ──► cb_objects sector/geo rollup
   └── general      ──► [Conversation Agent]    ──► LLaMA 70B
        │
   [Critic/Validator Node] ──► Hallucination check + Confidence scoring
        │
   [Telemetry Node] ──► route | latency | retries | db_queried | confidence
        │
     [END]
```

## Key Components

| Component | Technology | Extension |
|---|---|---|
| Orchestration | LangGraph StateGraph | 9 nodes, 7 routes (was 8/5) |
| Dense Retrieval | FAISS + HuggingFace | Extended with 3k+ startup narratives |
| Sparse Retrieval | BM25Okapi | Extended corpus |
| Fusion | Reciprocal Rank Fusion | k=60 |
| Structured Query | Text-to-SQL V2 | Dual-DB + retry loop (max 2) |
| Crunchbase DB | SQLite 9-table relational | 18 indexes, WAL journal |
| Anomaly Detection | Z-score peer-group | Ops + Funding + Acquisition |
| Investor Agent | SQL + RAG hybrid | cb_funds + investor corpus |
| Ecosystem Agent | Sector/Geo rollup SQL | cb_objects multi-dim |
| Forecasting | Linear regression | KPI + CB funding trends |
| Validation | Critic LLM Node | Fail-open, confidence penalty |
| Telemetry | Extended TelemetryRecord | retries + agent_type + db_queried |
| LLM Backbone | Groq (LLaMA 70B + 8B) | Low-latency inference |
| Dashboard | Gradio + Plotly | 6 tabs (was 4) |
            """)

print("✅ Gradio dashboard defined — 6 tabs.")
print("\n🚀 Launch with: dashboard.launch(share=True)")


/tmp/ipykernel_514/1171326736.py:183: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_514/1171326736.py:183: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


✅ Gradio dashboard defined — 6 tabs.

🚀 Launch with: dashboard.launch(share=True)


## 26. Pre-warm & Launch

In [34]:
# ── Pre-warm with sample queries so telemetry panel has data ──────────────────
print("🔥 Pre-warming ARIA with sample queries...")
warmup_queries = [
    ("Which country has the highest average startup valuation?",   "analytics"),
    ("What are the best practices for RAG architecture?",          "rag"),
    ("Are there any anomalies in the ML Inference service?",       "anomaly"),
    ("Which VC funds raised the most capital?",                    "investor"),
    ("What sectors have the highest acquisition exit rates?",      "ecosystem"),
]
for q, expected_route in warmup_queries:
    r = query_platform(q, thread_id="warmup")
    print(f"  [{expected_route.upper():<12}] ✅ done")

print(f"\n📈 Telemetry records loaded: {len(telemetry_log)}")
print("\n" + "="*65)
print("🚀 LAUNCHING ARIA ENTERPRISE INTELLIGENCE DASHBOARD")
print("="*65)

dashboard.launch(share=True, server_name="0.0.0.0", server_port=7860)


🔥 Pre-warming ARIA with sample queries...
🔀 Route: [ECOSYSTEM] | query: 'Which country has the highest average startup valuation?...'
  ⚠️  Critic flagged: ['LACK OF SPECIFIC DATA', 'HALLUCINATION']
  📊 Telemetry | 4083ms | route:ecosystem | conf:0.70 | retries:0 | db:crunchbase
  [ANALYTICS   ] ✅ done
🔀 Route: [RAG] | query: 'What are the best practices for RAG architecture?...'
  ✅ Critic passed (confidence: 1.00)
  📊 Telemetry | 1401ms | route:rag | conf:1.00 | retries:0 | db:enterprise
  [RAG         ] ✅ done
🔀 Route: [ANOMALY] | query: 'Are there any anomalies in the ML Inference service?...'
  ⚠️  Critic flagged: ['SEVERITY ASSESSMENT NEEDS CLEARER METHODOLOGY', 'LACK OF METRICS FOR ACQUISITION CLUSTERING']
  📊 Telemetry | 3532ms | route:anomaly | conf:0.80 | retries:0 | db:both
  [ANOMALY     ] ✅ done
🔀 Route: [INVESTOR] | query: 'Which VC funds raised the most capital?...'
  ⚠️  Critic flagged: ['LACK OF CLEAR DATA TO SUPPORT CLAIMS', 'NO SQL ACCURACY TO EVALUATE']
  📊 Telemetr